In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0-pre — Environment verification
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, '-c', 'import depthcharge; print(depthcharge.__file__)'],
    capture_output=True, text=True
)
dc_path = result.stdout.strip()
print(f'depthcharge location: {dc_path}')

if 'site-packages' in dc_path:
    print('⚠ Editable install not active — installing now...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e',
         '/teamspace/studios/this_studio/depthcharge_changes',
         '--break-system-packages'],
        check=True
    )
    print('Done. Please RESTART the kernel and re-run this cell.')
else:
    print('✓ Editable install confirmed')

    dc_root = os.path.dirname(dc_path)

    analytes_path = os.path.join(dc_root, 'transformers', 'analytes.py')
    spectra_path  = os.path.join(dc_root, 'transformers', 'spectra.py')

    with open(analytes_path) as f:
        analytes_src = f.read()
    with open(spectra_path) as f:
        spectra_src = f.read()

    ok1 = 'flash_compatible' in analytes_src
    ok2 = 'new_zeros' in spectra_src

    print(f'  flash_compatible in analytes.py : {"✓" if ok1 else "✗ MISSING"}')
    print(f'  new_zeros in spectra.py         : {"✓" if ok2 else "✗ MISSING"}')

    if not (ok1 and ok2):
        raise RuntimeError(
            f'Branch changes not detected.\n'
            f'  analytes.py path: {analytes_path}\n'
            f'  spectra.py path : {spectra_path}'
        )
    print('\nReady → proceed to Cell 0 (NAR patch)')

depthcharge location: /teamspace/studios/this_studio/depthcharge_changes/depthcharge/__init__.py
✓ Editable install confirmed
  flash_compatible in analytes.py : ✓
  new_zeros in spectra.py         : ✓

Ready → proceed to Cell 0 (NAR patch)


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0 — NAR Patch (flash_compatible variant)
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, warnings
warnings.filterwarnings('ignore')

import torch
from casanovo.denovo.transformers import PeptideDecoder
from casanovo.denovo.model import Spec2Pep
from depthcharge.transformers import AnalyteTransformerDecoder

import depthcharge as _dc
print(f'depthcharge: {_dc.__file__}')
if 'site-packages' in _dc.__file__:
    raise RuntimeError(
        'Editable install not active.\n'
        'Run: pip install -e /teamspace/studios/this_studio/depthcharge_changes --break-system-packages'
    )
print('  ✓ local branch confirmed (not site-packages)\n')

# ── PATCH 1 — PeptideDecoder.embed ────────────────────────────────────
_ar_embed_original = AnalyteTransformerDecoder.embed

def _nar_embed(self, tokens, *args,
               memory,
               memory_key_padding_mask=None,
               memory_mask=None,
               tgt_mask=None,
               flash_compatible=False,
               _orig=_ar_embed_original,
               **kwargs):
    return _orig(
        self, tokens, *args,
        memory=memory,
        memory_key_padding_mask=memory_key_padding_mask,
        memory_mask=memory_mask,
        flash_compatible=True,
        **kwargs,
    )

assert _ar_embed_original is not _nar_embed, (
    'BUG: captured original is already the patch — restart kernel.')
PeptideDecoder.embed = _nar_embed

# ── PATCH 2 — Spec2Pep._forward_step ─────────────────────────────────
def _nar_forward_step(self, batch):
    mzs, ints, precursors, seqs = self._process_batch(batch)
    dev = self.device
    mzs = mzs.to(dev); ints = ints.to(dev); precursors = precursors.to(dev)
    memories, mem_masks = self.encoder(mzs, ints)
    zero_tokens = (torch.zeros_like(seqs.to(dev)) if seqs is not None
                   else torch.zeros((mzs.shape[0], self.max_peptide_len),
                                    dtype=torch.long, device=dev))
    scores = self.decoder(tokens=zero_tokens, memory=memories,
                          memory_key_padding_mask=mem_masks,
                          precursors=precursors)
    return scores, seqs
Spec2Pep._forward_step = _nar_forward_step

# ── PATCH 3 — Spec2Pep.forward ───────────────────────────────────────
def _nar_forward(self, batch): return self._forward_step(batch)
Spec2Pep.forward = _nar_forward

# ── Verification ──────────────────────────────────────────────────────
import inspect as _inspect

_checks = {
    'PeptideDecoder.embed → _nar_embed'           : PeptideDecoder.embed is _nar_embed,
    'Spec2Pep._forward_step → _nar_forward_step'  : Spec2Pep._forward_step is _nar_forward_step,
    'Spec2Pep.forward → _nar_forward'             : Spec2Pep.forward is _nar_forward,
    'flash_compatible named param in _nar_embed'  : 'flash_compatible' in _inspect.signature(_nar_embed).parameters,
    'zero_tokens var in _nar_forward_step code'   : 'zero_tokens' in _nar_forward_step.__code__.co_varnames,
}
print('── NAR Patch Status ──────────────────────────────────────────')
for k, v in _checks.items():
    print(f'  {k:50s}: {"✓" if v else "✗ FAILED"}')
print('──────────────────────────────────────────────────────────────')
if not all(_checks.values()):
    raise RuntimeError('One or more patches failed.')
print('\nNAR patches applied ✓  (flash_compatible variant)')

depthcharge: /teamspace/studios/this_studio/depthcharge_changes/depthcharge/__init__.py
  ✓ local branch confirmed (not site-packages)

── NAR Patch Status ──────────────────────────────────────────
  PeptideDecoder.embed → _nar_embed                 : ✓
  Spec2Pep._forward_step → _nar_forward_step        : ✓
  Spec2Pep.forward → _nar_forward                   : ✓
  flash_compatible named param in _nar_embed        : ✓
  zero_tokens var in _nar_forward_step code         : ✓
──────────────────────────────────────────────────────────────

NAR patches applied ✓  (flash_compatible variant)


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Setup + Data + Model
# CHANGES vs previous version:
#   • Added _launch_counts()  → counts cudaLaunchKernel AND cudaGraphLaunch
#     (manual CUDA Graph replay shows up as cudaGraphLaunch, not cudaLaunchKernel)
#   • Added NARUnified module → encoder+decoder as ONE traceable forward,
#     the object needed for both fullgraph compile and manual graph capture
# ═══════════════════════════════════════════════════════════════════════
import os, time, threading
import numpy as np, pandas as pd, datetime
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from torch.profiler import profile, ProfilerActivity, schedule, record_function
from torch.nn.attention import SDPBackend, sdpa_kernel
from contextlib import contextmanager
from tqdm import tqdm

WORK_DIR    = '/teamspace/studios/this_studio/nar_profiling'
RESULTS_DIR = '/teamspace/studios/this_studio/profiling_after_depthcharge_changes/results'
os.chdir(WORK_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs('results', exist_ok=True)
print(f'Working directory : {os.getcwd()}')
print(f'Results directory : {RESULTS_DIR}')

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME   = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0
BF16_DTYPE     = torch.bfloat16
BF16_SUPPORTED = torch.cuda.is_bf16_supported() if DEVICE == 'cuda' else False

N_SUBSET         = 6000
N_TIMING_SPECTRA = 5000
BATCH_SIZES      = [1, 8, 32, 128, 512]
N_WARMUP_BATCHES = 10
N_PEAKS          = 150
PROF_WARMUP      = 20
PROF_ACTIVE      = 50

def _sync():
    if DEVICE == 'cuda': torch.cuda.synchronize()

@contextmanager
def _bf16_ctx():
    with torch.autocast(device_type='cuda', dtype=BF16_DTYPE):
        yield

@contextmanager
def _bf16_ctx_nocache():
    # cache_enabled=False is REQUIRED for manual CUDA Graph capture:
    # autocast's weight cache allocates tensors lazily, which is illegal
    # inside a capture region.
    with torch.autocast(device_type='cuda', dtype=BF16_DTYPE, cache_enabled=False):
        yield

@contextmanager
def _fp32_ctx():
    yield

def _mark_step():
    if DEVICE == 'cuda':
        torch.compiler.cudagraph_mark_step_begin()

def pad_or_select_to_fixed_peaks(mzs, ints, n=N_PEAKS):
    bs, L = mzs.shape
    if L == n: return mzs, ints, 0
    if L < n:
        return (torch.nn.functional.pad(mzs,  (0, n - L)),
                torch.nn.functional.pad(ints, (0, n - L)), 0)
    idx = ints.topk(n, dim=1).indices
    return mzs.gather(1, idx), ints.gather(1, idx), bs

def _start_gpu_monitor():
    samples, stop = [], threading.Event()
    def _fn():
        import subprocess as sp
        while not stop.is_set():
            r = sp.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used',
                        '--format=csv,noheader,nounits'],
                       capture_output=True, text=True)
            if r.returncode == 0:
                try:
                    u, m = r.stdout.strip().split(', ')
                    samples.append((int(u), float(m)/1024))
                except Exception: pass
            time.sleep(0.5)
    threading.Thread(target=_fn, daemon=True).start()
    return samples, stop

def _detect_attention_kernel(store, label):
    if not (DEVICE == 'cuda' and store.get('avgs')): return f'{label}: no CUDA data'
    flash = next((e for e in store['avgs'] if 'flash_attention' in e.key), None)
    eff   = next((e for e in store['avgs'] if 'efficient_attention' in e.key), None)
    if flash and eff:
        msg = f'{label}: BOTH flash ({flash.count}) + efficient ({eff.count})'
    elif flash:
        msg = f'{label}: FlashAttention ACTIVE ✓  ({flash.key}, {flash.count} calls)'
    elif eff:
        msg = f'{label}: memory-efficient only (NOT Flash)  ({eff.count} calls)'
    else:
        msg = f'{label}: no attention kernel found (likely inside a replayed CUDA Graph)'
    print(msg); return msg

def _launch_counts(store, n_active=PROF_ACTIVE):
    """Return (cudaLaunchKernel/spec, cudaGraphLaunch/spec).
    A manually captured graph replays as ONE cudaGraphLaunch — the individual
    kernel launches disappear from the CPU side entirely. That is the win."""
    if not store.get('avgs'): return None, None
    k = next((e for e in store['avgs'] if e.key == 'cudaLaunchKernel'), None)
    g = next((e for e in store['avgs'] if 'cudaGraphLaunch' in e.key), None)
    return (k.count // n_active if k else 0), (g.count // n_active if g else 0)

def _launch_count(store, n_active=PROF_ACTIVE):
    k, _ = _launch_counts(store, n_active)
    return k

def _compiled_region_count(store):
    if not store.get('avgs'): return None
    return len([e for e in store['avgs'] if 'Torch-Compiled Region' in e.key])

def _save(obj, name):
    path = os.path.join(RESULTS_DIR, name)
    if hasattr(obj, 'savefig'):
        obj.savefig(path, dpi=150, bbox_inches='tight')
    else:
        obj.to_csv(path, index=False)
    print(f'Saved: {path}')
    return path

_tgt = lambda ms: 'MEETS ✓' if ms <= 10 else f'FAILS ({ms:.1f}ms)'

MGF_FILE   = 'multi-enzyme-simple.test.mgf'
SUBSET_MGF = 'subset_profile.mgf'
LANCE_DIR  = '.lance_cache'

if not os.path.exists(MGF_FILE):
    raise FileNotFoundError(f'{MGF_FILE} not found in {WORK_DIR}')
print(f'{MGF_FILE}  ({os.path.getsize(MGF_FILE)/1e6:.1f} MB)')

print('Spectra : 106,933')
print('Charge  : +1 to +8 | +2: 40,614  +3: 39,146')
print('m/z     : 301.2 – 1604.3')
print('Peaks   : 123 avg  (min 6, max 950)')

if not os.path.exists(SUBSET_MGF):
    raise FileNotFoundError(f'{SUBSET_MGF} not found in {WORK_DIR}')
print(f'Reusing: {SUBSET_MGF}  ({os.path.getsize(SUBSET_MGF)/1e6:.1f} MB)')

from casanovo.denovo import ModelRunner
from casanovo.denovo.model import Spec2Pep
from casanovo.denovo.dataloaders import DeNovoDataModule
from casanovo.config import Config
from casanovo.casanovo import _get_model_weights
import appdirs

config = Config(None)
cache_dir = Path(appdirs.user_cache_dir('casanovo', False, opinion=False))
model_path = _get_model_weights(cache_dir)
print(f'Model checkpoint: {model_path}')

runner = ModelRunner(config, model_path)
runner.initialize_tokenizer()
runner.initialize_model(train=False)
model = runner.model.eval().to(DEVICE)
MODEL_MAX_CHARGE = getattr(model, 'max_charge', config.max_charge)

print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params | '
      f'max_peptide_len={model.max_peptide_len} | max_charge={MODEL_MAX_CHARGE}')

import inspect as _inspect
from casanovo.denovo.transformers import PeptideDecoder
assert PeptideDecoder.embed is _nar_embed, 'NAR patch lost — re-run Cell 0!'
ok_fc = 'flash_compatible=True' in _inspect.getsource(_nar_embed)
print(f'NAR patch active ✓ | flash_compatible=True in patch: {"✓" if ok_fc else "✗"}')


# ═══════════════════════════════════════════════════════════════════════
# NEW — Unified encoder+decoder module
# This is the "single region" equivalent of Spec2Pep.forward MINUS the
# CPU-side _process_batch() data prep. We deliberately exclude
# _process_batch because it is pure Python/CPU tensor marshalling that
# can never be graph-captured — including it would guarantee a break and
# tell us nothing. Everything from here on is pure GPU work.
# ═══════════════════════════════════════════════════════════════════════
class NARUnified(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.encoder = m.encoder
        self.decoder = m.decoder
        self.max_peptide_len = m.max_peptide_len

    def forward(self, mzs, ints, precursors, tokens):
        memory, mem_mask = self.encoder(mzs, ints)
        return self.decoder(tokens=tokens, memory=memory,
                            memory_key_padding_mask=mem_mask,
                            precursors=precursors)

unified = NARUnified(model).eval().to(DEVICE)
print('NARUnified module created (encoder → decoder, single forward) ✓')

# Pre-allocated zero-token buffers, one per batch size (avoids a fresh
# allocation on every iteration and gives manual capture a stable address)
ZERO_TOKENS = {bs: torch.zeros((bs, model.max_peptide_len),
                               dtype=torch.long, device=DEVICE)
               for bs in BATCH_SIZES}

_dm = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=1,
    tokenizer=runner.tokenizer,
    max_charge=MODEL_MAX_CHARGE,
    n_workers=0,
)
_dm.setup(stage='test', annotated=False)
_b              = next(iter(_dm.predict_dataloader()))
_mz, _int0, _pr, _ = model._process_batch(_b)
print(f'First batch mzs={_mz.shape}  precs={_pr.shape} ✓')
print(f'Subset ready: {N_SUBSET} spectra  (timing target: {N_TIMING_SPECTRA})')

print(f'\nDevice : {DEVICE} | {GPU_NAME} | VRAM: {TOTAL_VRAM:.1f} GB')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda} | BF16: {BF16_SUPPORTED}')
print(f'Batch sizes: {BATCH_SIZES}')

Working directory : /teamspace/studios/this_studio/nar_profiling
Results directory : /teamspace/studios/this_studio/profiling_after_depthcharge_changes/results
multi-enzyme-simple.test.mgf  (300.9 MB)
Spectra : 106,933
Charge  : +1 to +8 | +2: 40,614  +3: 39,146
m/z     : 301.2 – 1604.3
Peaks   : 123 avg  (min 6, max 950)
Reusing: subset_profile.mgf  (16.9 MB)


Checkpoint directory not set in ModelRunner, no checkpoint files will be saved.
Configured residue(s) not in model alphabet: Q[Deamidated], N[Deamidated], [Ammonia-loss]-, [+25.980265]-, C[Carbamidomethyl], [Acetyl]-, [Carbamyl]-, M[Oxidation]


Model checkpoint: /home/zeus/.cache/casanovo/casanovo_v5_0_0_v5_0_0.ckpt
Model: 47.9M params | max_peptide_len=100 | max_charge=4
NAR patch active ✓ | flash_compatible=True in patch: ✓
NARUnified module created (encoder → decoder, single forward) ✓


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

First batch mzs=torch.Size([1, 42])  precs=torch.Size([1, 3]) ✓
Subset ready: 6000 spectra  (timing target: 5000)

Device : cuda | NVIDIA L4 | VRAM: 23.7 GB
PyTorch: 2.7.1+cu128 | CUDA: 12.8 | BF16: True
Batch sizes: [1, 8, 32, 128, 512]


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1B  [REVISED] — Disable nn.TransformerEncoder nested-tensor fast path
#
# ROOT CAUSE (one line, three failures — confirmed by 3B, 4B, 4C):
#   torch/nn/modules/transformer.py:457
#     ) and not torch._nested_tensor_from_mask_left_aligned(
#           src, src_key_padding_mask.logical_not())
#   Returns a Python bool derived from DEVICE data →
#     • Dynamo can't trace it   → encoder graph break (compile gave 1.01×)
#     • Forces a host sync      → illegal during CUDA graph capture
#
# WHY THE PREVIOUS VERIFICATION FALSELY FAILED
#   It compared the FULL encoder memory tensor, padding included.
#   The nested path zero-fills padded slots on scatter-back; the dense
#   path computes them normally. Those slots are masked out of
#   cross-attention by memory_key_padding_mask and never affect output.
#   With 42 real peaks padded to 150, 108/151 positions were padding —
#   hence max_diff 7.02 on memory but only 8.8e-06 on decoder scores.
#
# CORRECT CRITERIA (all three checked below):
#   1. src_key_padding_mask identical between paths
#   2. encoder memory equal at VALID (non-padded) positions
#   3. decoder scores + argmax token predictions equal, over many spectra
#
#   Exact bit-equality is NOT expected: packed vs dense GEMMs use
#   different reduction orders. FP32 noise ~1e-5 is correct and expected.
#
# Cell is idempotent — safe to re-run.
#
# NOTEBOOK-ONLY. If validated, the permanent fix is one kwarg in
# depthcharge spectra.py:  nn.TransformerEncoder(..., enable_nested_tensor=False)
# ═══════════════════════════════════════════════════════════════════════
import torch.nn as _nn

TOL_VALID  = 1e-3     # encoder memory, valid positions (FP32 kernel noise)
TOL_SCORES = 1e-3     # decoder logits
N_EQUIV    = 50       # spectra for the equivalence sweep

_encoders = [(n or '<root>', m) for n, m in model.named_modules()
             if isinstance(m, _nn.TransformerEncoder)]
if not _encoders:
    raise RuntimeError('No nn.TransformerEncoder found — cannot apply fix.')

def _set_fastpath(enabled: bool):
    for _, m in _encoders:
        m.use_nested_tensor = enabled
        m.mask_check        = enabled

# ── Equivalence sweep data ────────────────────────────────────────────
_dm_fx = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                          eval_batch_size=1, tokenizer=runner.tokenizer,
                          max_charge=MODEL_MAX_CHARGE, n_workers=0)
_dm_fx.setup(stage='test', annotated=False)

_equiv_batches = []
for _fb in _dm_fx.predict_dataloader():
    _m, _i, _p, _ = model._process_batch(_fb)
    _m = _m.to(DEVICE); _i = _i.to(DEVICE); _p = _p.to(DEVICE)
    _m, _i, _ = pad_or_select_to_fixed_peaks(_m, _i)
    _equiv_batches.append((_m, _i, _p))
    if len(_equiv_batches) >= N_EQUIV: break
print(f'Equivalence sweep: {len(_equiv_batches)} bs=1 spectra')

_tok1 = ZERO_TOKENS[1]

def _run_once(mz, it, pr):
    with torch.no_grad():
        mem, msk = model.encoder(mz, it)
        out = model.decoder(tokens=_tok1, memory=mem,
                            memory_key_padding_mask=msk, precursors=pr)
    return mem.clone(), msk.clone(), out.clone()

# ── Reference: fast path ON (original PyTorch default) ────────────────
_set_fastpath(True)
_ref = [_run_once(*b) for b in _equiv_batches]
_sync()

# ── Apply fix: fast path OFF ──────────────────────────────────────────
_set_fastpath(False)
print('\n── nn.TransformerEncoder fast path disabled ──────────────────')
for _n, _m in _encoders:
    print(f'  {_n:<45} use_nested_tensor=False  mask_check=False')
assert unified.encoder is model.encoder, 'unified must share modules with model'
print('  unified shares the patched modules ✓')

_new = [_run_once(*b) for b in _equiv_batches]
_sync()

# ── Verification ──────────────────────────────────────────────────────
_mask_mismatch = 0
_d_valid_max, _d_pad_max, _d_scores_max = 0.0, 0.0, 0.0
_argmax_disagree, _n_valid_tot, _n_pad_tot = 0, 0, 0

for (rm, rk, ro), (nm, nk, no) in zip(_ref, _new):
    if not torch.equal(rk, nk):
        _mask_mismatch += 1

    valid = ~rk                                   # True = real peak / latent
    _n_valid_tot += int(valid.sum()); _n_pad_tot += int((~valid).sum())

    if valid.any():
        d = (rm[valid] - nm[valid]).abs().max().item()
        _d_valid_max = max(_d_valid_max, d)
    if (~valid).any():
        d = (rm[~valid] - nm[~valid]).abs().max().item()
        _d_pad_max = max(_d_pad_max, d)

    _d_scores_max = max(_d_scores_max, (ro - no).abs().max().item())
    if not torch.equal(ro.argmax(-1), no.argmax(-1)):
        _argmax_disagree += 1

print(f'\n── Numerical equivalence (fast path ON vs OFF, {len(_ref)} spectra) ──')
print(f'  src_key_padding_mask mismatches : {_mask_mismatch} / {len(_ref)}')
print(f'  positions: {_n_valid_tot} valid, {_n_pad_tot} padded '
      f'({100*_n_pad_tot/max(_n_valid_tot+_n_pad_tot,1):.0f}% padding)')
print(f'  encoder memory @ VALID positions : {_d_valid_max:.3e}   (must be < {TOL_VALID:.0e})')
print(f'  encoder memory @ PAD positions   : {_d_pad_max:.3e}   (IGNORED — masked downstream)')
print(f'  decoder scores                   : {_d_scores_max:.3e}   (must be < {TOL_SCORES:.0e})')
print(f'  argmax token disagreements       : {_argmax_disagree} / {len(_ref)}')

_pass = (_mask_mismatch == 0
         and _d_valid_max  < TOL_VALID
         and _d_scores_max < TOL_SCORES
         and _argmax_disagree == 0)

print(f'\n  VERDICT : {"EQUIVALENT ✓ — safe to proceed" if _pass else "FAILED — do not proceed"}')
if _pass:
    print('  The padded-position difference is expected and harmless: the nested')
    print('  path zero-fills padding on scatter-back, the dense path computes it.')
    print('  Those positions are masked out by memory_key_padding_mask.')
print('─────────────────────────────────────────────────────────────')

with open('results/nested_tensor_fix.txt', 'w') as f:
    f.write(f'patched_modules={[n for n,_ in _encoders]}\n'
            f'n_spectra={len(_ref)}\n'
            f'mask_mismatches={_mask_mismatch}\n'
            f'encoder_valid_max_diff={_d_valid_max}\n'
            f'encoder_pad_max_diff={_d_pad_max}  (ignored, masked downstream)\n'
            f'decoder_scores_max_diff={_d_scores_max}\n'
            f'argmax_disagreements={_argmax_disagree}\n'
            f'pass={_pass}\n')
print('Saved: results/nested_tensor_fix.txt')

if not _pass:
    _set_fastpath(True)
    raise RuntimeError('Fast-path change altered VALID outputs — reverted.')

print('\n→ Now run Cells 2, 3, 3B, 4, 4B, 4C, 5, 6, 7 in order.')

subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

Equivalence sweep: 50 bs=1 spectra

── nn.TransformerEncoder fast path disabled ──────────────────
  encoder.transformer_encoder                   use_nested_tensor=False  mask_check=False
  unified shares the patched modules ✓

── Numerical equivalence (fast path ON vs OFF, 50 spectra) ──
  src_key_padding_mask mismatches : 0 / 50
  positions: 4554 valid, 2996 padded (40% padding)
  encoder memory @ VALID positions : 2.768e-04   (must be < 1e-03)
  encoder memory @ PAD positions   : 7.068e+00   (IGNORED — masked downstream)
  decoder scores                   : 1.788e-04   (must be < 1e-03)
  argmax token disagreements       : 0 / 50

  VERDICT : EQUIVALENT ✓ — safe to proceed
  The padded-position difference is expected and harmless: the nested
  path zero-fills padding on scatter-back, the dense path computes it.
  Those positions are masked out by memory_key_padding_mask.
─────────────────────────────────────────────────────────────
Saved: results/nested_tensor_fix.txt

→ Now run Ce

In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — Baseline NAR FP32 Timing (eager)
# FIX: loop iterator renamed _loader_it (was _it, which shadowed the
#      intensity tensor from Cell 1)
# ═══════════════════════════════════════════════════════════════════════
_gpu_s, _gpu_stop = _start_gpu_monitor()
timing_fp32 = {}

for bs in BATCH_SIZES:
    print(f'\n══ FP32 Baseline  batch_size={bs:4d} ══')
    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)
    with torch.no_grad():
        for _w, _wb in enumerate(iter(_dm_bs.predict_dataloader())):
            if _w >= N_WARMUP_BATCHES: break
            _wm, _wi, _wp, _ = model._process_batch(_wb)
            _wm=_wm.to(DEVICE); _wi=_wi.to(DEVICE); _wp=_wp.to(DEVICE)
            _wme, _wmk = model.encoder(_wm, _wi)
            model.decoder(tokens=torch.zeros((_wm.shape[0], model.max_peptide_len),
                          dtype=torch.long, device=DEVICE),
                          memory=_wme, memory_key_padding_mask=_wmk, precursors=_wp)
    _sync()

    _t = {k: [] for k in ['fetch','h2d','enc','nar','write','total','tp']}
    _loader_it = iter(_dm_bs.predict_dataloader()); n_spec = 0
    pbar = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')
    while n_spec < N_TIMING_SPECTRA:
        _sync(); t0 = time.perf_counter()
        try: batch = next(_loader_it)
        except StopIteration: _loader_it = iter(_dm_bs.predict_dataloader()); batch = next(_loader_it)
        t_fetch = (time.perf_counter()-t0)*1000

        _sync(); t0 = time.perf_counter()
        mzs,ints,precs,_ = model._process_batch(batch)
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        _sync(); t_h2d=(time.perf_counter()-t0)*1000; ab=mzs.shape[0]

        with torch.no_grad():
            _sync(); t0=time.perf_counter()
            mem,mmk = model.encoder(mzs,ints)
            _sync(); t_enc=(time.perf_counter()-t0)*1000
            zt = torch.zeros((ab,model.max_peptide_len),dtype=torch.long,device=DEVICE)
            _sync(); t0=time.perf_counter()
            scores = model.decoder(tokens=zt,memory=mem,
                                   memory_key_padding_mask=mmk,precursors=precs)
            _sync(); t_nar=(time.perf_counter()-t0)*1000

        t0=time.perf_counter()
        pred=scores.argmax(dim=-1).cpu(); _=[{'tokens':t.tolist()} for t in pred]
        t_write=(time.perf_counter()-t0)*1000
        tt=t_fetch+t_h2d+t_enc+t_nar+t_write
        for k,v in zip(['fetch','h2d','enc','nar','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_enc/ab,t_nar/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec+=ab; pbar.update(ab)
        if n_spec>=N_TIMING_SPECTRA: break
    pbar.close()

    p = lambda a,q: float(np.percentile(a,q))
    timing_fp32[bs] = {'n_spec':n_spec,'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),
        'enc':np.mean(_t['enc']),'nar':np.mean(_t['nar']),'write':np.mean(_t['write']),
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s=timing_fp32[bs]
    print(f'  total={s["total"]:.2f}ms  enc={s["enc"]:.2f}ms  nar={s["nar"]:.2f}ms  tp={s["tp"]:.1f}spec/s')
    if DEVICE=='cuda': torch.cuda.empty_cache()

_gpu_stop.set(); time.sleep(1.0)
gpu_util_fp32=np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_fp32=np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

b1=timing_fp32[1]; rb=b1['raw']
df_stage_fp32 = pd.DataFrame([
    {'Stage':'DataLoader fetch','mean_ms':b1['fetch'],'p50_ms':np.percentile(rb['fetch'],50),'p95_ms':np.percentile(rb['fetch'],95)},
    {'Stage':'H2D transfer',    'mean_ms':b1['h2d'],  'p50_ms':np.percentile(rb['h2d'],50),  'p95_ms':np.percentile(rb['h2d'],95)},
    {'Stage':'SpectrumEncoder', 'mean_ms':b1['enc'],  'p50_ms':np.percentile(rb['enc'],50),  'p95_ms':np.percentile(rb['enc'],95)},
    {'Stage':'NAR Decoder',     'mean_ms':b1['nar'],  'p50_ms':np.percentile(rb['nar'],50),  'p95_ms':np.percentile(rb['nar'],95)},
    {'Stage':'Output write',    'mean_ms':b1['write'],'p50_ms':np.percentile(rb['write'],50),'p95_ms':np.percentile(rb['write'],95)},
    {'Stage':'TOTAL',           'mean_ms':b1['total'],'p50_ms':b1['p50'],                   'p95_ms':b1['p95']},
]).round(3)
df_tp_fp32 = pd.DataFrame([{'batch_size':bs,'total_ms':timing_fp32[bs]['total'],
    'tp_spec_s':timing_fp32[bs]['tp'],'enc_ms':timing_fp32[bs]['enc'],
    'nar_ms':timing_fp32[bs]['nar'],'p50':timing_fp32[bs]['p50'],
    'p95':timing_fp32[bs]['p95']} for bs in BATCH_SIZES]).round(3)
print(f'\n── Stage breakdown (FP32, bs=1, {b1["n_spec"]} spectra) ──')
print(df_stage_fp32.to_string(index=False))
print(f'\n── Throughput (FP32) ──\n{df_tp_fp32.to_string(index=False)}')
print(f'\nGPU util: {gpu_util_fp32:.0f}%  |  Peak VRAM: {gpu_vram_fp32:.2f} GB')
print(f'10ms target (bs=1): {_tgt(b1["total"])}')
df_stage_fp32.to_csv('results/fp32_stage_bs1.csv',index=False)
df_tp_fp32.to_csv('results/fp32_throughput.csv',index=False)


══ FP32 Baseline  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [01:36<00:00, 52.08spec/s]


  total=18.82ms  enc=5.38ms  nar=11.63ms  tp=53.9spec/s

══ FP32 Baseline  batch_size=   8 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

...

  bs=128: 5120spec [00:11, 428.65spec/s]                        

  total=2.32ms  enc=1.04ms  nar=1.07ms  tp=431.4spec/s

══ FP32 Baseline  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=512: 5120spec [00:12, 407.01spec/s]                        


  total=2.45ms  enc=1.09ms  nar=1.18ms  tp=407.6spec/s

── Stage breakdown (FP32, bs=1, 5000 spectra) ──
           Stage  mean_ms  p50_ms  p95_ms
DataLoader fetch    1.447   1.406   1.933
    H2D transfer    0.248   0.236   0.343
 SpectrumEncoder    5.378   5.086   8.097
     NAR Decoder   11.628  11.084  15.765
    Output write    0.123   0.117   0.169
           TOTAL   18.824  18.001  25.551

── Throughput (FP32) ──
 batch_size  total_ms  tp_spec_s  enc_ms  nar_ms    p50    p95
          1    18.824     53.901   5.378  11.628 18.001 25.551
          8     2.886    351.414   0.855   1.604  2.735  3.807
         32     2.247    445.346   0.982   0.996  2.236  2.350
        128     2.319    431.370   1.036   1.071  2.302  2.416
        512     2.454    407.632   1.090   1.175  2.450  2.506

GPU util: 40%  |  Peak VRAM: 3.42 GB
10ms target (bs=1): FAILS (18.8ms)


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — BF16 + FlashAttention Timing
# ═══════════════════════════════════════════════════════════════════════
assert BF16_SUPPORTED, 'BF16 not supported on this GPU.'

_probe_dm = DeNovoDataModule(
    lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
    eval_batch_size=1, tokenizer=runner.tokenizer,
    max_charge=MODEL_MAX_CHARGE, n_workers=0,
)
_probe_dm.setup(stage='test', annotated=False)
_probe_batch = next(iter(_probe_dm.predict_dataloader()))
_probe_mz, _probe_int, _probe_pr, _ = model._process_batch(_probe_batch)

_pb_mz, _pb_it, _pb_pr = _probe_mz.to(DEVICE), _probe_int.to(DEVICE), _probe_pr.to(DEVICE)
_pz = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)
FLASH_ACCEPTED = False
_flash_err = ''
try:
    with torch.no_grad(), _bf16_ctx():
        with sdpa_kernel(backends=[SDPBackend.FLASH_ATTENTION]):
            _pme, _pmk = model.encoder(_pb_mz, _pb_it)
            model.decoder(tokens=_pz, memory=_pme,
                          memory_key_padding_mask=_pmk, precursors=_pb_pr)
    FLASH_ACCEPTED = True
except RuntimeError as _e:
    _flash_err = str(_e)[:120]
_sync()

print('── FlashAttention probe (BF16 + flash_compatible=True) ──────')
if FLASH_ACCEPTED:
    print('  RESULT: Flash ACCEPTED ✓')
else:
    print(f'  RESULT: Flash rejected — {_flash_err}')
    print('  NOTE: cross-attention keeps memory_key_padding_mask → efficient kernel.')
print('─────────────────────────────────────────────────────────────\n')

_gpu_s, _gpu_stop = _start_gpu_monitor()
timing_bf16 = {}

for bs in BATCH_SIZES:
    print(f'\n══ BF16+Flash  batch_size={bs:4d} ══')
    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)
    with torch.no_grad(), _bf16_ctx():
        for _w,_wb in enumerate(iter(_dm_bs.predict_dataloader())):
            if _w>=N_WARMUP_BATCHES: break
            _wm,_wi,_wp,_ = model._process_batch(_wb)
            _wm=_wm.to(DEVICE); _wi=_wi.to(DEVICE); _wp=_wp.to(DEVICE)
            _wme,_wmk = model.encoder(_wm,_wi)
            model.decoder(tokens=torch.zeros((_wm.shape[0],model.max_peptide_len),
                          dtype=torch.long,device=DEVICE),
                          memory=_wme,memory_key_padding_mask=_wmk,precursors=_wp)
    _sync()

    _t={k:[] for k in ['fetch','h2d','enc','nar','write','total','tp']}
    _loader_it=iter(_dm_bs.predict_dataloader()); n_spec=0
    pbar=tqdm(total=N_TIMING_SPECTRA,desc=f'  bs={bs}',unit='spec')
    while n_spec<N_TIMING_SPECTRA:
        _sync(); t0=time.perf_counter()
        try: batch=next(_loader_it)
        except StopIteration: _loader_it=iter(_dm_bs.predict_dataloader()); batch=next(_loader_it)
        t_fetch=(time.perf_counter()-t0)*1000
        _sync(); t0=time.perf_counter()
        mzs,ints,precs,_=model._process_batch(batch)
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        _sync(); t_h2d=(time.perf_counter()-t0)*1000; ab=mzs.shape[0]
        with torch.no_grad(), _bf16_ctx():
            _sync(); t0=time.perf_counter()
            mem,mmk=model.encoder(mzs,ints)
            _sync(); t_enc=(time.perf_counter()-t0)*1000
            zt=torch.zeros((ab,model.max_peptide_len),dtype=torch.long,device=DEVICE)
            _sync(); t0=time.perf_counter()
            scores=model.decoder(tokens=zt,memory=mem,
                                 memory_key_padding_mask=mmk,precursors=precs)
            _sync(); t_nar=(time.perf_counter()-t0)*1000
        t0=time.perf_counter()
        pred=scores.argmax(dim=-1).cpu(); _=[{'tokens':t.tolist()} for t in pred]
        t_write=(time.perf_counter()-t0)*1000
        tt=t_fetch+t_h2d+t_enc+t_nar+t_write
        for k,v in zip(['fetch','h2d','enc','nar','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_enc/ab,t_nar/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec+=ab; pbar.update(ab)
        if n_spec>=N_TIMING_SPECTRA: break
    pbar.close()
    p=lambda a,q: float(np.percentile(a,q))
    timing_bf16[bs]={'n_spec':n_spec,'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),
        'enc':np.mean(_t['enc']),'nar':np.mean(_t['nar']),'write':np.mean(_t['write']),
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s=timing_bf16[bs]
    print(f'  total={s["total"]:.2f}ms  enc={s["enc"]:.2f}ms  nar={s["nar"]:.2f}ms  tp={s["tp"]:.1f}spec/s')
    print(f'  vs FP32: {timing_fp32[bs]["total"]/max(s["total"],0.001):.2f}×')
    if DEVICE=='cuda': torch.cuda.empty_cache()

_gpu_stop.set(); time.sleep(1.0)
gpu_util_bf16=np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_bf16=np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

c1=timing_bf16[1]; rc=c1['raw']
df_stage_bf16=pd.DataFrame([
    {'Stage':'DataLoader fetch',   'mean_ms':c1['fetch'],'p50_ms':np.percentile(rc['fetch'],50),'p95_ms':np.percentile(rc['fetch'],95)},
    {'Stage':'H2D transfer',       'mean_ms':c1['h2d'],  'p50_ms':np.percentile(rc['h2d'],50),  'p95_ms':np.percentile(rc['h2d'],95)},
    {'Stage':'SpectrumEncoder(BF16)','mean_ms':c1['enc'],'p50_ms':np.percentile(rc['enc'],50),  'p95_ms':np.percentile(rc['enc'],95)},
    {'Stage':'NAR Decoder(BF16+Flash)','mean_ms':c1['nar'],'p50_ms':np.percentile(rc['nar'],50),'p95_ms':np.percentile(rc['nar'],95)},
    {'Stage':'Output write',       'mean_ms':c1['write'],'p50_ms':np.percentile(rc['write'],50),'p95_ms':np.percentile(rc['write'],95)},
    {'Stage':'TOTAL',              'mean_ms':c1['total'],'p50_ms':c1['p50'],                   'p95_ms':c1['p95']},
]).round(3)
df_tp_bf16=pd.DataFrame([{'batch_size':bs,'total_ms':timing_bf16[bs]['total'],
    'tp_spec_s':timing_bf16[bs]['tp'],'enc_ms':timing_bf16[bs]['enc'],
    'nar_ms':timing_bf16[bs]['nar'],'p50':timing_bf16[bs]['p50'],
    'p95':timing_bf16[bs]['p95'],'vs_fp32':f"{timing_fp32[bs]['total']/max(timing_bf16[bs]['total'],0.001):.2f}x"}
    for bs in BATCH_SIZES]).round(3)
print(f'\n── Stage breakdown (BF16+Flash, bs=1) ──\n{df_stage_bf16.to_string(index=False)}')
print(f'\n── Throughput (BF16+Flash) ──\n{df_tp_bf16.to_string(index=False)}')
print(f'\nGPU util: {gpu_util_bf16:.0f}%  |  Peak VRAM: {gpu_vram_bf16:.2f} GB')
print(f'10ms target (bs=1): {_tgt(c1["total"])}')
_save(df_stage_bf16, 'bf16_stage_bs1.csv')
_save(df_tp_bf16, 'bf16_throughput.csv')
df_stage_bf16.to_csv('results/bf16_stage_bs1.csv',index=False)
df_tp_bf16.to_csv('results/bf16_throughput.csv',index=False)

subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

── FlashAttention probe (BF16 + flash_compatible=True) ──────
  RESULT: Flash rejected — No available kernel. Aborting execution.
  NOTE: cross-attention keeps memory_key_padding_mask → efficient kernel.
─────────────────────────────────────────────────────────────


══ BF16+Flash  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [02:17<00:00, 36.48spec/s]


  total=26.68ms  enc=9.61ms  nar=15.14ms  tp=37.9spec/s
  vs FP32: 0.71×

══ BF16+Flash  batch_size=   8 ══


...

  bs=128: 5120spec [00:05, 943.82spec/s]                        


  total=1.04ms  enc=0.36ms  nar=0.46ms  tp=959.8spec/s
  vs FP32: 2.22×

══ BF16+Flash  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=512: 5120spec [00:05, 962.96spec/s]                        


  total=1.03ms  enc=0.40ms  nar=0.44ms  tp=967.6spec/s
  vs FP32: 2.37×

── Stage breakdown (BF16+Flash, bs=1) ──
                  Stage  mean_ms  p50_ms  p95_ms
       DataLoader fetch    1.528   1.479   2.013
           H2D transfer    0.251   0.240   0.347
  SpectrumEncoder(BF16)    9.615   9.069  13.843
NAR Decoder(BF16+Flash)   15.142  14.518  19.204
           Output write    0.142   0.136   0.194
                  TOTAL   26.677  25.553  34.933

── Throughput (BF16+Flash) ──
 batch_size  total_ms  tp_spec_s  enc_ms  nar_ms    p50    p95 vs_fp32
          1    26.677     37.936   9.615  15.142 25.553 34.933   0.71x
          8     3.795    267.812   1.294   2.063  3.580  5.015   0.76x
         32     1.130    901.536   0.342   0.538  1.054  1.493   1.99x
        128     1.044    959.777   0.361   0.465  1.021  1.136   2.22x
        512     1.034    967.616   0.397   0.444  1.033  1.067   2.37x

GPU util: 20%  |  Peak VRAM: 2.80 GB
10ms target (bs=1): FAILS (26.7ms)
Saved: /teams

In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3B  [REVISED] — Graph-break diagnosis via torch._dynamo.explain
#
# FIX vs previous version:
#   Previously the verdict used exp.graph_break_count, which reported 0 for
#   the encoder even though a break WAS printed. Reason: dynamo computes
#   graph_break_count = graph_count - 1, and a TRAILING break still leaves
#   graph_count == 1. len(exp.break_reasons) is the reliable signal.
#   Verdict now uses break_reasons.
#
# Run AFTER Cell 1B so this reflects the nested-tensor fix.
# NOTE: explain() calls torch._dynamo.reset() internally → run BEFORE Cell 4.
# ═══════════════════════════════════════════════════════════════════════
import torch._dynamo
torch._dynamo.config.cache_size_limit = 32

_dm_ex = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                          eval_batch_size=1, tokenizer=runner.tokenizer,
                          max_charge=MODEL_MAX_CHARGE, n_workers=0)
_dm_ex.setup(stage='test', annotated=False)
_exb = next(iter(_dm_ex.predict_dataloader()))
_ex_mz, _ex_int, _ex_pr, _ = model._process_batch(_exb)
_ex_mz=_ex_mz.to(DEVICE); _ex_int=_ex_int.to(DEVICE); _ex_pr=_ex_pr.to(DEVICE)
_ex_mz,_ex_int,_ = pad_or_select_to_fixed_peaks(_ex_mz,_ex_int)
_ex_tok = ZERO_TOKENS[1]

_explain_log = []

def _explain(fn, args, label):
    print(f'\n{"="*70}\n── dynamo.explain: {label}\n{"="*70}')
    _explain_log.append(f'\n=== {label} ===')
    try:
        with torch.no_grad(), _bf16_ctx():
            exp = torch._dynamo.explain(fn)(*args)
    except Exception as e:
        msg = f'  explain FAILED: {type(e).__name__}: {str(e)[:300]}'
        print(msg); _explain_log.append(msg); return None, None

    reasons  = list(getattr(exp, 'break_reasons', None) or [])
    n_real   = len(reasons)                       # ← authoritative count
    n_graph  = getattr(exp, 'graph_count', None)
    n_ops    = getattr(exp, 'op_count', None)
    n_report = getattr(exp, 'graph_break_count', None)

    head = (f'  graphs={n_graph}  ops={n_ops}  '
            f'break_reasons={n_real}  (dynamo graph_break_count={n_report})')
    print(head); _explain_log.append(head)

    if n_real == 0:
        m = '  → NO graph breaks ✓  (fully traceable)'
        print(m); _explain_log.append(m)
    for i, r in enumerate(reasons, 1):
        rsn = getattr(r, 'reason', str(r))
        m = f'\n  [break {i}] {rsn}'
        print(m); _explain_log.append(m)
        for fr in list(getattr(r, 'user_stack', None) or [])[-5:]:
            loc = f'      {fr.filename}:{fr.lineno}  in {fr.name}'
            print(loc); _explain_log.append(loc)
            if getattr(fr, 'line', None):
                src = f'          {fr.line.strip()}'
                print(src); _explain_log.append(src)
    return exp, n_real

_exp_enc, _nb_enc = _explain(model.encoder, (_ex_mz, _ex_int),
                             'SpectrumEncoder (alone)')

_enc_mem, _enc_msk = model.encoder(_ex_mz, _ex_int)
_exp_dec, _nb_dec = _explain(
    lambda tok, mem, mmk, pr: model.decoder(
        tokens=tok, memory=mem, memory_key_padding_mask=mmk, precursors=pr),
    (_ex_tok, _enc_mem, _enc_msk, _ex_pr),
    'PeptideDecoder (alone)')

_exp_uni, _nb_uni = _explain(unified, (_ex_mz, _ex_int, _ex_pr, _ex_tok),
                             'NARUnified (encoder + decoder, ONE function)')

_bk_enc, _bk_dec, _bk_uni = _nb_enc, _nb_dec, _nb_uni
_gr_uni = getattr(_exp_uni, 'graph_count', None) if _exp_uni else None

print(f'\n{"="*70}\n── VERDICT ──────────────────────────────────────────────────')
print(f'  encoder real breaks : {_bk_enc}')
print(f'  decoder real breaks : {_bk_dec}')
print(f'  unified real breaks : {_bk_uni}   (unified graphs: {_gr_uni})')
if _bk_uni == 0:
    verdict = ('  → CLEAN ✓ nested-tensor fast-path fix (Cell 1B) removed the break.\n'
               '    Encoder should now receive CUDA Graphs like the decoder already does.\n'
               '    Expect the ~9.8 ms encoder stage to drop sharply in Cell 4.')
else:
    verdict = (f'  → {_bk_uni} break(s) still present — see file:line above.\n'
               '    If this still names _nested_tensor_from_mask_left_aligned,\n'
               '    Cell 1B did not apply (check it found an nn.TransformerEncoder).')
print(verdict); print('='*70)
_explain_log.append('\n=== VERDICT ===\n' + verdict)

with open('results/dynamo_explain.txt','w') as f:
    f.write('\n'.join(_explain_log))
print('\nSaved: results/dynamo_explain.txt')

torch._dynamo.reset()
if DEVICE=='cuda': torch.cuda.empty_cache()
print('dynamo reset — ready for Cell 4')

subset_profile.mgf: 0 spectra [00:00, ? spectra/s]


── dynamo.explain: SpectrumEncoder (alone)
  graphs=1  ops=132  break_reasons=0  (dynamo graph_break_count=0)
  → NO graph breaks ✓  (fully traceable)

── dynamo.explain: PeptideDecoder (alone)
  graphs=1  ops=188  break_reasons=0  (dynamo graph_break_count=0)
  → NO graph breaks ✓  (fully traceable)

── dynamo.explain: NARUnified (encoder + decoder, ONE function)
  graphs=1  ops=320  break_reasons=0  (dynamo graph_break_count=0)
  → NO graph breaks ✓  (fully traceable)

── VERDICT ──────────────────────────────────────────────────
  encoder real breaks : 0
  decoder real breaks : 0
  unified real breaks : 0   (unified graphs: 1)
  → CLEAN ✓ nested-tensor fast-path fix (Cell 1B) removed the break.
    Encoder should now receive CUDA Graphs like the decoder already does.
    Expect the ~9.8 ms encoder stage to drop sharply in Cell 4.

Saved: results/dynamo_explain.txt
dynamo reset — ready for Cell 4


In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — torch.compile (SPLIT: encoder and decoder separately) + BF16
# This is the existing best-known configuration (~12.9 ms at bs=1).
# Kept unchanged so the new variants in 4B/4C are directly comparable.
# ═══════════════════════════════════════════════════════════════════════
torch._dynamo.config.cache_size_limit = 32

compiled_encoder = torch.compile(model.encoder, mode='reduce-overhead')
compiled_decoder = torch.compile(model.decoder, mode='reduce-overhead')
print('compiled_encoder / compiled_decoder created (mode=reduce-overhead)')

_gpu_s, _gpu_stop = _start_gpu_monitor()
timing_compiled = {}

for bs in BATCH_SIZES:
    print(f'\n══ Compiled(split)+BF16+Flash  batch_size={bs:4d} ══')
    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)

    print(f'  Compiling + capturing for bs={bs}…')
    _t0_compile = time.perf_counter()
    with torch.no_grad(), _bf16_ctx():
        for _w,_wb in enumerate(iter(_dm_bs.predict_dataloader())):
            if _w>=N_WARMUP_BATCHES: break
            _wm,_wi,_wp,_ = model._process_batch(_wb)
            _wm=_wm.to(DEVICE); _wi=_wi.to(DEVICE); _wp=_wp.to(DEVICE)
            if _wm.shape[0] != bs: continue
            _wm,_wi,_ = pad_or_select_to_fixed_peaks(_wm,_wi)
            _mark_step()
            _wme,_wmk = compiled_encoder(_wm,_wi)
            compiled_decoder(tokens=ZERO_TOKENS[bs],
                             memory=_wme,memory_key_padding_mask=_wmk,precursors=_wp)
    _sync()
    compile_s = time.perf_counter()-_t0_compile
    print(f'  Compile+capture done in {compile_s:.1f}s (excluded from timings)')

    _t={k:[] for k in ['fetch','h2d','enc','nar','write','total','tp']}
    _loader_it=iter(_dm_bs.predict_dataloader()); n_spec=0
    pbar=tqdm(total=N_TIMING_SPECTRA,desc=f'  bs={bs}',unit='spec')
    while n_spec<N_TIMING_SPECTRA:
        _sync(); t0=time.perf_counter()
        try: batch=next(_loader_it)
        except StopIteration: _loader_it=iter(_dm_bs.predict_dataloader()); batch=next(_loader_it)
        t_fetch=(time.perf_counter()-t0)*1000
        _sync(); t0=time.perf_counter()
        mzs,ints,precs,_=model._process_batch(batch)
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        mzs,ints,_ntrunc=pad_or_select_to_fixed_peaks(mzs,ints)
        _sync(); t_h2d=(time.perf_counter()-t0)*1000; ab=mzs.shape[0]
        if ab != bs:      # partial trailing batch → skip (avoids recompile)
            continue
        with torch.no_grad(), _bf16_ctx():
            _mark_step()
            _sync(); t0=time.perf_counter()
            mem,mmk=compiled_encoder(mzs,ints)
            _sync(); t_enc=(time.perf_counter()-t0)*1000
            _sync(); t0=time.perf_counter()
            scores=compiled_decoder(tokens=ZERO_TOKENS[bs],memory=mem,
                                    memory_key_padding_mask=mmk,precursors=precs)
            _sync(); t_nar=(time.perf_counter()-t0)*1000
        t0=time.perf_counter()
        pred=scores.argmax(dim=-1).cpu(); _=[{'tokens':t.tolist()} for t in pred]
        t_write=(time.perf_counter()-t0)*1000
        tt=t_fetch+t_h2d+t_enc+t_nar+t_write
        for k,v in zip(['fetch','h2d','enc','nar','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_enc/ab,t_nar/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec+=ab; pbar.update(ab)
        if n_spec>=N_TIMING_SPECTRA: break
    pbar.close()
    p=lambda a,q:float(np.percentile(a,q))
    timing_compiled[bs]={'n_spec':n_spec,'compile_s':compile_s,
        'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),
        'enc':np.mean(_t['enc']),'nar':np.mean(_t['nar']),'write':np.mean(_t['write']),
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s=timing_compiled[bs]
    print(f'  total={s["total"]:.2f}ms  enc={s["enc"]:.2f}ms  nar={s["nar"]:.2f}ms  '
          f'tp={s["tp"]:.1f}spec/s  vs FP32: {timing_fp32[bs]["total"]/max(s["total"],0.001):.2f}×')
    if DEVICE=='cuda': torch.cuda.empty_cache()

_gpu_stop.set(); time.sleep(1.0)
gpu_util_comp=np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_comp=np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

e1=timing_compiled[1]; re=e1['raw']
df_stage_comp=pd.DataFrame([
    {'Stage':'DataLoader fetch',       'mean_ms':e1['fetch'],'p50_ms':np.percentile(re['fetch'],50),'p95_ms':np.percentile(re['fetch'],95)},
    {'Stage':'H2D+fixed-peak pad',     'mean_ms':e1['h2d'],  'p50_ms':np.percentile(re['h2d'],50),  'p95_ms':np.percentile(re['h2d'],95)},
    {'Stage':'Encoder(Compiled+BF16)', 'mean_ms':e1['enc'],  'p50_ms':np.percentile(re['enc'],50),  'p95_ms':np.percentile(re['enc'],95)},
    {'Stage':'Decoder(Compiled+Flash)','mean_ms':e1['nar'],  'p50_ms':np.percentile(re['nar'],50),  'p95_ms':np.percentile(re['nar'],95)},
    {'Stage':'Output write',           'mean_ms':e1['write'],'p50_ms':np.percentile(re['write'],50),'p95_ms':np.percentile(re['write'],95)},
    {'Stage':'TOTAL',                  'mean_ms':e1['total'],'p50_ms':e1['p50'],                   'p95_ms':e1['p95']},
]).round(3)
df_tp_comp=pd.DataFrame([{'batch_size':bs,'total_ms':timing_compiled[bs]['total'],
    'tp_spec_s':timing_compiled[bs]['tp'],'enc_ms':timing_compiled[bs]['enc'],
    'nar_ms':timing_compiled[bs]['nar'],'compile_s':round(timing_compiled[bs]['compile_s'],1),
    'vs_fp32':f"{timing_fp32[bs]['total']/max(timing_compiled[bs]['total'],0.001):.2f}x"}
    for bs in BATCH_SIZES]).round(3)
print(f'\n── Stage breakdown (Compiled split, bs=1) ──\n{df_stage_comp.to_string(index=False)}')
print(f'\n── Throughput (Compiled split) ──\n{df_tp_comp.to_string(index=False)}')
print(f'\nGPU util: {gpu_util_comp:.0f}%  |  Peak VRAM: {gpu_vram_comp:.2f} GB')
print(f'10ms target (bs=1): {_tgt(e1["total"])}')
df_stage_comp.to_csv('results/compiled_stage_bs1.csv',index=False)
df_tp_comp.to_csv('results/compiled_throughput.csv',index=False)

compiled_encoder / compiled_decoder created (mode=reduce-overhead)

══ Compiled(split)+BF16+Flash  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compiling + capturing for bs=1…


W0818 14:22:09.847000 91010 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/_inductor/utils.py:1250] [0/0] Not enough SMs to use max_autotune_gemm mode


  Compile+capture done in 19.8s (excluded from timings)


...

  total=0.90ms  enc=0.28ms  nar=0.41ms  tp=1107.4spec/s  vs FP32: 2.56×

══ Compiled(split)+BF16+Flash  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compiling + capturing for bs=512…
  Compile+capture done in 4.7s (excluded from timings)


  bs=512: 5120spec [00:04, 1185.43spec/s]                        


  total=0.84ms  enc=0.30ms  nar=0.33ms  tp=1193.8spec/s  vs FP32: 2.92×

── Stage breakdown (Compiled split, bs=1) ──
                  Stage  mean_ms  p50_ms  p95_ms
       DataLoader fetch    1.219   1.160   1.649
     H2D+fixed-peak pad    0.320   0.311   0.478
 Encoder(Compiled+BF16)    1.241   1.216   1.385
Decoder(Compiled+Flash)    1.662   1.632   1.843
           Output write    0.133   0.124   0.178
                  TOTAL    4.575   4.426   5.479

── Throughput (Compiled split) ──
 batch_size  total_ms  tp_spec_s  enc_ms  nar_ms  compile_s vs_fp32
          1     4.575    220.107   1.241   1.662       19.8   4.11x
          8     1.767    567.927   0.573   0.776       37.0   1.63x
         32     0.908   1106.583   0.299   0.336        1.6   2.47x
        128     0.904   1107.357   0.278   0.406        2.2   2.56x
        512     0.840   1193.794   0.296   0.333        4.7   2.92x

GPU util: 24%  |  Peak VRAM: 2.62 GB
10ms target (bs=1): MEETS ✓


In [9]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4B  [REVISED] — Unified single-region compile (fullgraph=True)
#
# FIXES vs previous version:
#   • torch._dynamo.reset() BEFORE the probe, so a stale compiled entry
#     from Cell 4 can't mask the fullgraph result.
#   • Warm-up now filters partial batches BEFORE the shape check, so a
#     short trailing batch can't trigger a silent recompile.
#   • With the Cell 1B fix in place, fullgraph=True is EXPECTED to succeed.
#     If it still fails, the printed error is the remaining blocker.
# ═══════════════════════════════════════════════════════════════════════
torch._dynamo.reset()
torch._dynamo.config.cache_size_limit = 32

FULLGRAPH_OK  = False
FULLGRAPH_ERR = ''

try:
    _probe_uni = torch.compile(unified, mode='reduce-overhead', fullgraph=True)
    with torch.no_grad(), _bf16_ctx():
        _mark_step()
        _ = _probe_uni(_ex_mz, _ex_int, _ex_pr, ZERO_TOKENS[1])
    _sync()
    FULLGRAPH_OK = True
    compiled_unified = _probe_uni
    print('── fullgraph=True ACCEPTED ✓ — encoder+decoder trace as ONE graph')
    print('   (this is the direct payoff of the Cell 1B nested-tensor fix)')
except Exception as _e:
    FULLGRAPH_ERR = f'{type(_e).__name__}: {str(_e)[:800]}'
    print('── fullgraph=True REJECTED — a real graph break remains:')
    print(f'   {FULLGRAPH_ERR}')
    print('   Falling back to fullgraph=False so we still get a number.')
    torch._dynamo.reset()
    compiled_unified = torch.compile(unified, mode='reduce-overhead')

with open('results/fullgraph_status.txt','w') as f:
    f.write(f'fullgraph_ok={FULLGRAPH_OK}\n{FULLGRAPH_ERR}\n')

_gpu_s, _gpu_stop = _start_gpu_monitor()
timing_unified = {}

for bs in BATCH_SIZES:
    tag = 'fullgraph' if FULLGRAPH_OK else 'fallback'
    print(f'\n══ Unified({tag})+BF16+Flash  batch_size={bs:4d} ══')
    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)

    print(f'  Compiling + capturing for bs={bs}…')
    _t0c = time.perf_counter(); _nwarm = 0
    with torch.no_grad(), _bf16_ctx():
        for _wb in iter(_dm_bs.predict_dataloader()):
            if _nwarm >= N_WARMUP_BATCHES: break
            _wm,_wi,_wp,_ = model._process_batch(_wb)
            if _wm.shape[0] != bs: continue          # skip partial → no recompile
            _wm=_wm.to(DEVICE); _wi=_wi.to(DEVICE); _wp=_wp.to(DEVICE)
            _wm,_wi,_ = pad_or_select_to_fixed_peaks(_wm,_wi)
            _mark_step()
            compiled_unified(_wm,_wi,_wp,ZERO_TOKENS[bs])
            _nwarm += 1
    _sync()
    compile_s = time.perf_counter()-_t0c
    print(f'  Compile+capture done in {compile_s:.1f}s ({_nwarm} warmup batches)')

    _t={k:[] for k in ['fetch','h2d','fwd','write','total','tp']}
    _loader_it=iter(_dm_bs.predict_dataloader()); n_spec=0
    pbar=tqdm(total=N_TIMING_SPECTRA,desc=f'  bs={bs}',unit='spec')
    while n_spec<N_TIMING_SPECTRA:
        _sync(); t0=time.perf_counter()
        try: batch=next(_loader_it)
        except StopIteration: _loader_it=iter(_dm_bs.predict_dataloader()); batch=next(_loader_it)
        t_fetch=(time.perf_counter()-t0)*1000
        _sync(); t0=time.perf_counter()
        mzs,ints,precs,_=model._process_batch(batch)
        if mzs.shape[0] != bs: continue
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        mzs,ints,_=pad_or_select_to_fixed_peaks(mzs,ints)
        _sync(); t_h2d=(time.perf_counter()-t0)*1000; ab=mzs.shape[0]
        with torch.no_grad(), _bf16_ctx():
            _mark_step()
            _sync(); t0=time.perf_counter()
            scores = compiled_unified(mzs,ints,precs,ZERO_TOKENS[bs])
            _sync(); t_fwd=(time.perf_counter()-t0)*1000
        t0=time.perf_counter()
        pred=scores.argmax(dim=-1).cpu(); _=[{'tokens':t.tolist()} for t in pred]
        t_write=(time.perf_counter()-t0)*1000
        tt=t_fetch+t_h2d+t_fwd+t_write
        for k,v in zip(['fetch','h2d','fwd','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_fwd/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec+=ab; pbar.update(ab)
        if n_spec>=N_TIMING_SPECTRA: break
    pbar.close()
    p=lambda a,q:float(np.percentile(a,q))
    timing_unified[bs]={'n_spec':n_spec,'compile_s':compile_s,
        'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),'fwd':np.mean(_t['fwd']),
        'write':np.mean(_t['write']),'enc':np.nan,'nar':np.nan,
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s=timing_unified[bs]
    print(f'  total={s["total"]:.2f}ms  fused_fwd={s["fwd"]:.2f}ms  tp={s["tp"]:.1f}spec/s')
    print(f'  vs FP32: {timing_fp32[bs]["total"]/max(s["total"],0.001):.2f}×   '
          f'vs Compiled(split): {timing_compiled[bs]["total"]/max(s["total"],0.001):.2f}×')
    if DEVICE=='cuda': torch.cuda.empty_cache()

_gpu_stop.set(); time.sleep(1.0)
gpu_util_uni=np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_uni=np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

u1=timing_unified[1]; ru=u1['raw']
df_stage_uni=pd.DataFrame([
    {'Stage':'DataLoader fetch',   'mean_ms':u1['fetch'],'p50_ms':np.percentile(ru['fetch'],50),'p95_ms':np.percentile(ru['fetch'],95)},
    {'Stage':'H2D+fixed-peak pad', 'mean_ms':u1['h2d'],  'p50_ms':np.percentile(ru['h2d'],50),  'p95_ms':np.percentile(ru['h2d'],95)},
    {'Stage':'Enc+Dec (1 region)', 'mean_ms':u1['fwd'],  'p50_ms':np.percentile(ru['fwd'],50),  'p95_ms':np.percentile(ru['fwd'],95)},
    {'Stage':'Output write',       'mean_ms':u1['write'],'p50_ms':np.percentile(ru['write'],50),'p95_ms':np.percentile(ru['write'],95)},
    {'Stage':'TOTAL',              'mean_ms':u1['total'],'p50_ms':u1['p50'],                   'p95_ms':u1['p95']},
]).round(3)
df_tp_uni=pd.DataFrame([{'batch_size':bs,'total_ms':timing_unified[bs]['total'],
    'tp_spec_s':timing_unified[bs]['tp'],'fwd_ms':timing_unified[bs]['fwd'],
    'compile_s':round(timing_unified[bs]['compile_s'],1),
    'vs_fp32':f"{timing_fp32[bs]['total']/max(timing_unified[bs]['total'],0.001):.2f}x",
    'vs_split':f"{timing_compiled[bs]['total']/max(timing_unified[bs]['total'],0.001):.2f}x"}
    for bs in BATCH_SIZES]).round(3)
print(f'\n── Stage breakdown (Unified, bs=1) ──\n{df_stage_uni.to_string(index=False)}')
print(f'\n── Throughput (Unified) ──\n{df_tp_uni.to_string(index=False)}')
print(f'\nGPU util: {gpu_util_uni:.0f}%  |  Peak VRAM: {gpu_vram_uni:.2f} GB')
print(f'10ms target (bs=1): {_tgt(u1["total"])}')
df_stage_uni.to_csv('results/unified_stage_bs1.csv',index=False)
df_tp_uni.to_csv('results/unified_throughput.csv',index=False)

── fullgraph=True ACCEPTED ✓ — encoder+decoder trace as ONE graph
   (this is the direct payoff of the Cell 1B nested-tensor fix)

══ Unified(fullgraph)+BF16+Flash  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compiling + capturing for bs=1…
  Compile+capture done in 0.9s (10 warmup batches)


  bs=1: 100%|██████████| 5000/5000 [00:23<00:00, 210.84spec/s]


  total=4.47ms  fused_fwd=2.76ms  tp=225.8spec/s
  vs FP32: 4.21×   vs Compiled(split): 1.02×

══ Unified(fullgraph)+BF16+Flash  batch_size=   8 ══


...

  total=0.89ms  fused_fwd=0.67ms  tp=1121.4spec/s
  vs FP32: 2.60×   vs Compiled(split): 1.01×

══ Unified(fullgraph)+BF16+Flash  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compiling + capturing for bs=512…
  Compile+capture done in 4.5s (10 warmup batches)


  bs=512: 5120spec [00:04, 1211.63spec/s]                        


  total=0.82ms  fused_fwd=0.62ms  tp=1217.8spec/s
  vs FP32: 2.99×   vs Compiled(split): 1.02×

── Stage breakdown (Unified, bs=1) ──
             Stage  mean_ms  p50_ms  p95_ms
  DataLoader fetch    1.250   1.165   1.712
H2D+fixed-peak pad    0.327   0.311   0.492
Enc+Dec (1 region)    2.756   2.691   3.107
      Output write    0.137   0.126   0.187
             TOTAL    4.469   4.281   5.406

── Throughput (Unified) ──
 batch_size  total_ms  tp_spec_s  fwd_ms  compile_s vs_fp32 vs_split
          1     4.469    225.841   2.756        0.9   4.21x    1.02x
          8     1.766    568.524   1.323       45.5   1.63x    1.00x
         32     0.919   1095.045   0.617        1.3   2.45x    0.99x
        128     0.893   1121.418   0.672        2.0   2.60x    1.01x
        512     0.822   1217.765   0.622        4.5   2.99x    1.02x

GPU util: 27%  |  Peak VRAM: 2.76 GB
10ms target (bs=1): MEETS ✓


In [10]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4C  [REVISED] — Manual torch.cuda.CUDAGraph capture
#
# WHAT WENT WRONG PREVIOUSLY (two separate bugs):
#
# BUG 1 — the capture itself.
#   nn.TransformerEncoder called _nested_tensor_from_mask_left_aligned(),
#   which reads device data into a Python bool → a HOST SYNC. CUDA graph
#   capture forbids host syncs → "operation not permitted when stream is
#   capturing". FIXED by Cell 1B (use_nested_tensor=False).
#
# BUG 2 — the error handler made things worse.
#   The except block called torch.cuda.empty_cache() while a capture was
#   still marked underway, tripping:
#     captures_underway.empty() INTERNAL ASSERT FAILED ... CUDACachingAllocator
#   That masked the real error and corrupted the allocator. The handler
#   below never touches the allocator during a failed capture.
#
# NEW: PRE-FLIGHT SYNC DETECTION.
#   Before entering capture we run the forward under
#   torch.cuda.set_sync_debug_mode("error"), which RAISES on any implicit
#   device sync without touching the graph API. If a sync is detected we
#   skip capture entirely — so a bad model state can no longer poison the
#   CUDA context and force a kernel restart.
#
# IF A CAPTURE STILL FAILS: the CUDA context may be unrecoverable.
#   CAPTURE_POISONED is set, remaining batch sizes are skipped, and you
#   should restart the kernel and re-run from Cell 0.
# ═══════════════════════════════════════════════════════════════════════
MANUAL_OK  = {}
MANUAL_ERR = {}
timing_manual = {}
_manual_graphs = {}
CAPTURE_POISONED = False

_sync()
if DEVICE=='cuda': torch.cuda.empty_cache()

# Shared memory pool so graphs across batch sizes reuse allocations
_graph_pool = torch.cuda.graph_pool_handle() if DEVICE=='cuda' else None


def _preflight_sync_check(mz, it, pr, tok):
    """Run the forward with implicit-sync detection ON.
    Raises nothing; returns (ok, message). Safe — never enters capture."""
    try:
        torch.cuda.synchronize()
        torch.cuda.set_sync_debug_mode('error')
    except Exception as e:
        return True, f'(sync-debug unavailable: {type(e).__name__}) — proceeding'
    try:
        with torch.no_grad(), _bf16_ctx_nocache():
            _ = unified(mz, it, pr, tok)
        ok, msg = True, 'no implicit device syncs detected ✓'
    except Exception as e:
        ok, msg = False, f'{type(e).__name__}: {str(e)[:300]}'
    finally:
        try: torch.cuda.set_sync_debug_mode('default')
        except Exception: pass
        try: torch.cuda.synchronize()
        except Exception: pass
    return ok, msg


def _capture_manual_graph(bs, mz0, it0, pr0):
    """Capture unified(...) into a CUDA graph. Raises on failure."""
    s_mz  = mz0.clone()
    s_int = it0.clone()
    s_pr  = pr0.clone()
    s_tok = torch.zeros((bs, model.max_peptide_len), dtype=torch.long, device=DEVICE)

    # Warm-up on a SIDE STREAM (mandatory before capture)
    side = torch.cuda.Stream()
    side.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(side):
        with torch.no_grad(), _bf16_ctx_nocache():
            for _ in range(5):
                _ = unified(s_mz, s_int, s_pr, s_tok)
    torch.cuda.current_stream().wait_stream(side)
    torch.cuda.synchronize()

    g = torch.cuda.CUDAGraph()
    with torch.no_grad(), _bf16_ctx_nocache():
        with torch.cuda.graph(g, pool=_graph_pool):
            s_out = unified(s_mz, s_int, s_pr, s_tok)
    torch.cuda.synchronize()
    return g, (s_mz, s_int, s_pr, s_tok), s_out


_gpu_s, _gpu_stop = _start_gpu_monitor()

for bs in BATCH_SIZES:
    print(f'\n══ Manual CUDA Graph + BF16  batch_size={bs:4d} ══')
    if CAPTURE_POISONED:
        print('  SKIPPED — a previous capture failed; CUDA context may be unusable.')
        MANUAL_OK[bs]=False; MANUAL_ERR[bs]='skipped (context poisoned)'
        continue

    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)

    _seed = None
    for _sb in _dm_bs.predict_dataloader():
        _m,_i,_p,_ = model._process_batch(_sb)
        if _m.shape[0] != bs: continue
        _m=_m.to(DEVICE); _i=_i.to(DEVICE); _p=_p.to(DEVICE)
        _m,_i,_ = pad_or_select_to_fixed_peaks(_m,_i)
        _seed = (_m,_i,_p); break
    if _seed is None:
        print(f'  skipped: no full batch of size {bs} in subset')
        MANUAL_OK[bs]=False; MANUAL_ERR[bs]='no full batch'
        continue

    # ── PRE-FLIGHT (safe: cannot poison the context) ──────────────────
    _pf_tok = torch.zeros((bs, model.max_peptide_len), dtype=torch.long, device=DEVICE)
    _pf_ok, _pf_msg = _preflight_sync_check(*_seed, _pf_tok)
    print(f'  Pre-flight sync check: {_pf_msg}')
    if not _pf_ok:
        print('  → An implicit device sync remains. NOT attempting capture')
        print('    (attempting anyway would corrupt the CUDA context).')
        print('    Check that Cell 1B applied; the sync is usually the')
        print('    nn.TransformerEncoder nested-tensor fast path.')
        MANUAL_OK[bs]=False; MANUAL_ERR[bs]=f'preflight sync: {_pf_msg}'
        continue

    print('  Capturing graph…')
    _t0c = time.perf_counter()
    try:
        g, (s_mz, s_int, s_pr, s_tok), s_out = _capture_manual_graph(bs, *_seed)
        MANUAL_OK[bs] = True
        _manual_graphs[bs] = (g, s_mz, s_int, s_pr, s_out)
    except Exception as _e:
        MANUAL_OK[bs]  = False
        MANUAL_ERR[bs] = f'{type(_e).__name__}: {str(_e)[:400]}'
        CAPTURE_POISONED = True
        print(f'  CAPTURE FAILED — {MANUAL_ERR[bs]}')
        print('  ⚠ NOT calling empty_cache() (that is what produced the')
        print('    "captures_underway INTERNAL ASSERT FAILED" crash before).')
        print('  ⚠ RESTART THE KERNEL and re-run from Cell 0 before trusting')
        print('    any further GPU numbers in this session.')
        continue
    capture_s = time.perf_counter()-_t0c
    print(f'  Capture OK ✓ in {capture_s:.1f}s')

    _t={k:[] for k in ['fetch','h2d','fwd','write','total','tp']}
    _loader_it=iter(_dm_bs.predict_dataloader()); n_spec=0
    pbar=tqdm(total=N_TIMING_SPECTRA,desc=f'  bs={bs}',unit='spec')
    while n_spec<N_TIMING_SPECTRA:
        _sync(); t0=time.perf_counter()
        try: batch=next(_loader_it)
        except StopIteration: _loader_it=iter(_dm_bs.predict_dataloader()); batch=next(_loader_it)
        t_fetch=(time.perf_counter()-t0)*1000
        _sync(); t0=time.perf_counter()
        mzs,ints,precs,_=model._process_batch(batch)
        if mzs.shape[0] != bs: continue
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        mzs,ints,_=pad_or_select_to_fixed_peaks(mzs,ints)
        _sync(); t_h2d=(time.perf_counter()-t0)*1000; ab=mzs.shape[0]

        # fused forward = static-buffer copy + ONE graph replay
        _sync(); t0=time.perf_counter()
        s_mz.copy_(mzs); s_int.copy_(ints); s_pr.copy_(precs)
        g.replay()
        _sync(); t_fwd=(time.perf_counter()-t0)*1000

        t0=time.perf_counter()
        pred=s_out.argmax(dim=-1).cpu(); _=[{'tokens':t.tolist()} for t in pred]
        t_write=(time.perf_counter()-t0)*1000
        tt=t_fetch+t_h2d+t_fwd+t_write
        for k,v in zip(['fetch','h2d','fwd','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_fwd/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec+=ab; pbar.update(ab)
        if n_spec>=N_TIMING_SPECTRA: break
    pbar.close()
    p=lambda a,q:float(np.percentile(a,q))
    timing_manual[bs]={'n_spec':n_spec,'compile_s':capture_s,
        'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),'fwd':np.mean(_t['fwd']),
        'write':np.mean(_t['write']),'enc':np.nan,'nar':np.nan,
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s=timing_manual[bs]
    print(f'  total={s["total"]:.2f}ms  replay={s["fwd"]:.2f}ms  tp={s["tp"]:.1f}spec/s')
    print(f'  vs FP32: {timing_fp32[bs]["total"]/max(s["total"],0.001):.2f}×   '
          f'vs Compiled(split): {timing_compiled[bs]["total"]/max(s["total"],0.001):.2f}×')

_gpu_stop.set(); time.sleep(1.0)
gpu_util_man=np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_man=np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

print('\n── Manual capture status per batch size ──')
for bs in BATCH_SIZES:
    print(f'  bs={bs:<5} {"OK ✓" if MANUAL_OK.get(bs) else "FAILED — "+str(MANUAL_ERR.get(bs,"?"))[:90]}')

if 1 in timing_manual:
    m1=timing_manual[1]; rm=m1['raw']
    df_stage_man=pd.DataFrame([
        {'Stage':'DataLoader fetch',   'mean_ms':m1['fetch'],'p50_ms':np.percentile(rm['fetch'],50),'p95_ms':np.percentile(rm['fetch'],95)},
        {'Stage':'H2D+fixed-peak pad', 'mean_ms':m1['h2d'],  'p50_ms':np.percentile(rm['h2d'],50),  'p95_ms':np.percentile(rm['h2d'],95)},
        {'Stage':'Graph replay (E+D)', 'mean_ms':m1['fwd'],  'p50_ms':np.percentile(rm['fwd'],50),  'p95_ms':np.percentile(rm['fwd'],95)},
        {'Stage':'Output write',       'mean_ms':m1['write'],'p50_ms':np.percentile(rm['write'],50),'p95_ms':np.percentile(rm['write'],95)},
        {'Stage':'TOTAL',              'mean_ms':m1['total'],'p50_ms':m1['p50'],                   'p95_ms':m1['p95']},
    ]).round(3)
    df_tp_man=pd.DataFrame([{'batch_size':bs,'total_ms':timing_manual[bs]['total'],
        'tp_spec_s':timing_manual[bs]['tp'],'replay_ms':timing_manual[bs]['fwd'],
        'vs_fp32':f"{timing_fp32[bs]['total']/max(timing_manual[bs]['total'],0.001):.2f}x",
        'vs_split':f"{timing_compiled[bs]['total']/max(timing_manual[bs]['total'],0.001):.2f}x"}
        for bs in BATCH_SIZES if bs in timing_manual]).round(3)
    print(f'\n── Stage breakdown (Manual graph, bs=1) ──\n{df_stage_man.to_string(index=False)}')
    print(f'\n── Throughput (Manual graph) ──\n{df_tp_man.to_string(index=False)}')
    print(f'\nGPU util: {gpu_util_man:.0f}%  |  Peak VRAM: {gpu_vram_man:.2f} GB')
    print(f'10ms target (bs=1): {_tgt(m1["total"])}')
    df_stage_man.to_csv('results/manual_graph_stage_bs1.csv',index=False)
    df_tp_man.to_csv('results/manual_graph_throughput.csv',index=False)
else:
    df_stage_man = None; df_tp_man = None
    print('\nManual graph capture did not succeed at bs=1 — see status above.')


══ Manual CUDA Graph + BF16  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Pre-flight sync check: RuntimeError: called a synchronizing CUDA operation
  → An implicit device sync remains. NOT attempting capture
    (attempting anyway would corrupt the CUDA context).
    Check that Cell 1B applied; the sync is usually the
    nn.TransformerEncoder nested-tensor fast path.

══ Manual CUDA Graph + BF16  batch_size=   8 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Pre-flight sync check: RuntimeError: called a synchronizing CUDA operation
  → An implicit device sync remains. NOT attempting capture
    (attempting anyway would corrupt the CUDA context).
    Check that Cell 1B applied; the sync is usually the
    nn.TransformerEncoder nested-tensor fast path.

══ Manual CUDA Graph + BF16  batch_size=  32 ══


...

  Pre-flight sync check: RuntimeError: called a synchronizing CUDA operation
  → An implicit device sync remains. NOT attempting capture
    (attempting anyway would corrupt the CUDA context).
    Check that Cell 1B applied; the sync is usually the
    nn.TransformerEncoder nested-tensor fast path.

══ Manual CUDA Graph + BF16  batch_size= 128 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Pre-flight sync check: RuntimeError: called a synchronizing CUDA operation
  → An implicit device sync remains. NOT attempting capture
    (attempting anyway would corrupt the CUDA context).
    Check that Cell 1B applied; the sync is usually the
    nn.TransformerEncoder nested-tensor fast path.

══ Manual CUDA Graph + BF16  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Pre-flight sync check: RuntimeError: called a synchronizing CUDA operation
  → An implicit device sync remains. NOT attempting capture
    (attempting anyway would corrupt the CUDA context).
    Check that Cell 1B applied; the sync is usually the
    nn.TransformerEncoder nested-tensor fast path.

── Manual capture status per batch size ──
  bs=1     FAILED — preflight sync: RuntimeError: called a synchronizing CUDA operation
  bs=8     FAILED — preflight sync: RuntimeError: called a synchronizing CUDA operation
  bs=32    FAILED — preflight sync: RuntimeError: called a synchronizing CUDA operation
  bs=128   FAILED — preflight sync: RuntimeError: called a synchronizing CUDA operation
  bs=512   FAILED — preflight sync: RuntimeError: called a synchronizing CUDA operation

Manual graph capture did not succeed at bs=1 — see status above.


In [11]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — torch.profiler across ALL variants
#   A) FP32 eager        B) BF16+Flash eager      C) Compiled (split)
#   D) Compiled unified  E) Manual CUDA Graph
#
# _run_prof now takes a single fused fwd(mz, it, pr, tok) callable so all
# five variants go through identical instrumentation.
#
# KEY METRIC: cudaLaunchKernel/spec vs cudaGraphLaunch/spec.
# A fully captured graph should show ~1 cudaGraphLaunch and near-zero
# cudaLaunchKernel — that is what "closing the dispatch gap" looks like.
# ═══════════════════════════════════════════════════════════════════════
ACTS = ([ProfilerActivity.CPU, ProfilerActivity.CUDA]
        if DEVICE == 'cuda' else [ProfilerActivity.CPU])
N_PROF = PROF_WARMUP + PROF_ACTIVE

print(f'Pre-fetching {N_PROF} bs=1 batches…')
_dm_prof = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                            eval_batch_size=1, tokenizer=runner.tokenizer,
                            max_charge=MODEL_MAX_CHARGE, n_workers=0)
_dm_prof.setup(stage='test', annotated=False)

_prof_batches, _prof_batches_fixed = [], []
for _pb in _dm_prof.predict_dataloader():
    _mz2, _in2, _pr2, _ = model._process_batch(_pb)
    _prof_batches.append((_mz2.to(DEVICE), _in2.to(DEVICE), _pr2.to(DEVICE)))
    _mzf, _inf, _ = pad_or_select_to_fixed_peaks(_mz2.to(DEVICE), _in2.to(DEVICE))
    _prof_batches_fixed.append((_mzf, _inf, _pr2.to(DEVICE)))
    if len(_prof_batches) >= N_PROF: break
while len(_prof_batches) < N_PROF:
    _prof_batches.extend(_prof_batches[:N_PROF-len(_prof_batches)])
    _prof_batches_fixed.extend(_prof_batches_fixed[:N_PROF-len(_prof_batches_fixed)])
print(f'Using {len(_prof_batches)} batches')

_zt_prof = ZERO_TOKENS[1]

def _split_fwd(enc, dec):
    def _f(mz, it, pr, tok):
        m, k = enc(mz, it)
        return dec(tokens=tok, memory=m, memory_key_padding_mask=k, precursors=pr)
    return _f

def _manual_fwd_factory(bs=1):
    if bs not in _manual_graphs: return None
    g, s_mz, s_int, s_pr, s_out = _manual_graphs[bs]
    def _f(mz, it, pr, tok):
        s_mz.copy_(mz); s_int.copy_(it); s_pr.copy_(pr)
        g.replay()
        return s_out
    return _f

def _run_prof(label, trace_path, txt_path, batches, ctx_fn, fwd,
              use_mark_step=False, n_warm=10):
    with torch.no_grad(), ctx_fn():
        for _mz2,_in2,_pr2 in batches[:n_warm]:
            if use_mark_step: _mark_step()
            fwd(_mz2,_in2,_pr2,_zt_prof)
    _sync()
    _store={}
    def _on_ready(p):
        p.export_chrome_trace(trace_path)
        _store['tbl']  = p.key_averages().table(sort_by='cpu_time_total', row_limit=12)
        _store['avgs'] = p.key_averages()
    with profile(activities=ACTS, record_shapes=True,
                 schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
                 on_trace_ready=_on_ready) as p:
        with torch.no_grad(), ctx_fn():
            for _mz2,_in2,_pr2 in batches:
                if use_mark_step: _mark_step()
                with record_function(label):
                    fwd(_mz2,_in2,_pr2,_zt_prof)
                _sync(); p.step()
    print(_store.get('tbl','(no data)'))
    with open(txt_path,'w') as f:
        f.write(f'{label}  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
        f.write('='*64+'\n'+str(_store.get('tbl','no data')))
    print(f'Chrome trace → {trace_path}')
    if DEVICE=='cuda': torch.cuda.synchronize(); torch.cuda.empty_cache()
    return _store

# ── A) Baseline FP32 ──────────────────────────────────────────────────
print('\n── A) FP32 (eager) ──────────────────────────────────────────')
_store_a = _run_prof('fp32', 'results/trace_fp32.json', 'results/profiler_fp32.txt',
                     _prof_batches, _fp32_ctx, _split_fwd(model.encoder, model.decoder))
_attn_a = _detect_attention_kernel(_store_a, 'A) FP32')
_nkern_a, _ngraph_a = _launch_counts(_store_a); _nreg_a = _compiled_region_count(_store_a)

# ── B) BF16 + Flash ───────────────────────────────────────────────────
print('\n── B) BF16 + Flash (eager) ──────────────────────────────────')
_store_b = _run_prof('bf16_flash', 'results/trace_bf16.json', 'results/profiler_bf16.txt',
                     _prof_batches, _bf16_ctx, _split_fwd(model.encoder, model.decoder))
_attn_b = _detect_attention_kernel(_store_b, 'B) BF16+Flash')
_nkern_b, _ngraph_b = _launch_counts(_store_b); _nreg_b = _compiled_region_count(_store_b)

# ── C) Compiled SPLIT ─────────────────────────────────────────────────
print('\n── C) Compiled (split enc/dec) + BF16 + Flash ───────────────')
_store_c = _run_prof('compiled_split', 'results/trace_compiled.json', 'results/profiler_compiled.txt',
                     _prof_batches_fixed, _bf16_ctx,
                     _split_fwd(compiled_encoder, compiled_decoder), use_mark_step=True)
_attn_c = _detect_attention_kernel(_store_c, 'C) Compiled(split)')
_nkern_c, _ngraph_c = _launch_counts(_store_c); _nreg_c = _compiled_region_count(_store_c)

# ── D) Compiled UNIFIED ───────────────────────────────────────────────
print('\n── D) Compiled UNIFIED (fullgraph) + BF16 + Flash ───────────')
_store_d = _run_prof('compiled_unified', 'results/trace_unified.json', 'results/profiler_unified.txt',
                     _prof_batches_fixed, _bf16_ctx,
                     lambda mz,it,pr,tok: compiled_unified(mz,it,pr,tok), use_mark_step=True)
_attn_d = _detect_attention_kernel(_store_d, 'D) Compiled(unified)')
_nkern_d, _ngraph_d = _launch_counts(_store_d); _nreg_d = _compiled_region_count(_store_d)

# ── E) Manual CUDA Graph ──────────────────────────────────────────────
_manual_fwd = _manual_fwd_factory(1)
if _manual_fwd is not None:
    print('\n── E) Manual CUDA Graph replay + BF16 ───────────────────────')
    _store_e = _run_prof('manual_cudagraph', 'results/trace_manual.json', 'results/profiler_manual.txt',
                         _prof_batches_fixed, _fp32_ctx, _manual_fwd)
    _attn_e = _detect_attention_kernel(_store_e, 'E) Manual graph')
    _nkern_e, _ngraph_e = _launch_counts(_store_e); _nreg_e = _compiled_region_count(_store_e)
else:
    _store_e={}; _attn_e='E) Manual graph: capture failed'
    _nkern_e=_ngraph_e=_nreg_e=None
    print('\n── E) SKIPPED (manual capture failed at bs=1) ───────────────')

# ── Summary table ─────────────────────────────────────────────────────
print('\n── Profiler Comparison Summary (per spectrum, bs=1) ──────────')
_hdr = f'{"Metric":<28}{"A:FP32":>10}{"B:BF16":>10}{"C:Split":>10}{"D:Unified":>11}{"E:Manual":>10}'
print(_hdr); print('-'*len(_hdr))
print(f'{"cudaLaunchKernel":<28}{str(_nkern_a):>10}{str(_nkern_b):>10}{str(_nkern_c):>10}{str(_nkern_d):>11}{str(_nkern_e):>10}')
print(f'{"cudaGraphLaunch":<28}{str(_ngraph_a):>10}{str(_ngraph_b):>10}{str(_ngraph_c):>10}{str(_ngraph_d):>11}{str(_ngraph_e):>10}')
print(f'{"Torch-Compiled Regions":<28}{str(_nreg_a):>10}{str(_nreg_b):>10}{str(_nreg_c):>10}{str(_nreg_d):>11}{str(_nreg_e):>10}')
print(f'\nfullgraph=True accepted: {"YES ✓" if FULLGRAPH_OK else "NO — "+FULLGRAPH_ERR[:80]}')

Pre-fetching 70 bs=1 batches…


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

Using 70 batches

── A) FP32 (eager) ──────────────────────────────────────────
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.39%       7.258ms       100.00%        1.866s      37.313ms       0.000us         0.00%     152.268ms       3.045ms            50  
                                                   fp32        26.96%     502.994ms        99.49%        1.856s      37.125ms  

In [12]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — Comparison plots across all successful variants
# ═══════════════════════════════════════════════════════════════════════
ALL_TIMINGS = {
    'fp32'    : timing_fp32,
    'bf16'    : timing_bf16,
    'compiled': timing_compiled,
    'unified' : timing_unified,
    'manual'  : timing_manual,
}
ALL_TIMINGS = {k:v for k,v in ALL_TIMINGS.items()
               if all(bs in v for bs in BATCH_SIZES)}

COLORS = {'fp32':'#D85A30','bf16':'#1D9E75','compiled':'#3A7FC1',
          'unified':'#8E44AD','manual':'#E8B12A'}
LABELS = {
    'fp32'    : 'Baseline FP32 (eager)',
    'bf16'    : 'BF16 + Flash (eager)',
    'compiled': 'Compiled SPLIT + BF16',
    'unified' : f'Compiled UNIFIED{" (fullgraph)" if FULLGRAPH_OK else ""} + BF16',
    'manual'  : 'Manual CUDA Graph + BF16',
}
STAGE_DFS = {'fp32':df_stage_fp32,'bf16':df_stage_bf16,'compiled':df_stage_comp,
             'unified':df_stage_uni,'manual':df_stage_man}

_xi   = list(range(len(BATCH_SIZES)))
_xlbl = [str(b) for b in BATCH_SIZES]

# ── Figure 1 — stage breakdown, one panel per variant ────────────────
_keys = [k for k in ALL_TIMINGS if STAGE_DFS.get(k) is not None]
fig1, axes = plt.subplots(1, len(_keys), figsize=(4.6*len(_keys), 5))
if len(_keys)==1: axes=[axes]
fig1.suptitle('Stage Breakdown (bs=1) — All Variants', fontweight='bold')
for ax, key in zip(axes, _keys):
    df = STAGE_DFS[key]
    sub = df[df['Stage']!='TOTAL']
    bars = ax.bar(range(len(sub)), sub['mean_ms'].values,
                  color=COLORS[key], edgecolor='none', width=0.55)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels([s.split('(')[0].strip()[:12] for s in sub['Stage']],
                       rotation=30, ha='right', fontsize=8)
    for b,v in zip(bars, sub['mean_ms'].values):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1, f'{v:.2f}',
                ha='center', fontsize=7)
    ax.set_title(f'{LABELS[key]}\nTotal: {ALL_TIMINGS[key][1]["total"]:.1f} ms', fontsize=9)
    ax.set_ylabel('ms / spectrum')
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/stage_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: results/stage_comparison.png')

# ── Figure 2 — throughput + latency vs batch size ────────────────────
fig2,(ax_tp,ax_lat)=plt.subplots(1,2,figsize=(15,5))
fig2.suptitle('NAR Performance vs Batch Size — All Variants', fontweight='bold')
for key,timing in ALL_TIMINGS.items():
    ax_tp.plot(_xi,[timing[bs]['tp'] for bs in BATCH_SIZES],'o-',
               color=COLORS[key],lw=2,ms=7,label=LABELS[key])
    ax_lat.plot(_xi,[timing[bs]['total'] for bs in BATCH_SIZES],'o-',
                color=COLORS[key],lw=2,ms=7,label=LABELS[key])
for ax,ylabel,title in [(ax_tp,'Throughput (spec/s)','Throughput vs Batch Size'),
                        (ax_lat,'ms / spectrum','Latency vs Batch Size')]:
    ax.set_xticks(_xi); ax.set_xticklabels(_xlbl)
    ax.set_xlabel('Batch size'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.spines[['top','right']].set_visible(False)
ax_lat.axhline(10,color='black',lw=1.5,ls=':')
ax_lat.axhline(35,color='purple',lw=1.2,ls='--')
ax_tp.legend(fontsize=7,frameon=False); ax_lat.legend(fontsize=7,frameon=False)
plt.tight_layout()
plt.savefig('results/throughput_comparison.png',dpi=150,bbox_inches='tight')
plt.show(); print('Saved: results/throughput_comparison.png')

# ── Figure 3 — latency histograms (bs=1) ─────────────────────────────
fig3,axes=plt.subplots(1,len(ALL_TIMINGS),figsize=(4.6*len(ALL_TIMINGS),5),sharey=True)
if len(ALL_TIMINGS)==1: axes=[axes]
fig3.suptitle('Latency Distribution (bs=1, 5000 spectra)',fontweight='bold')
for ax,(key,timing) in zip(axes,ALL_TIMINGS.items()):
    _raw=timing[1]['raw']['total']
    ax.hist(_raw,bins=30,color=COLORS[key],alpha=0.8,edgecolor='none')
    ax.axvline(np.mean(_raw),color='black',lw=2,ls='--',label=f'mean={np.mean(_raw):.1f}ms')
    ax.axvline(10,color='red',lw=1.5,ls=':',label='10ms target')
    ax.set_title(LABELS[key],fontsize=9); ax.set_xlabel('ms / spectrum')
    ax.legend(fontsize=7,frameon=False); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/latency_histograms.png',dpi=150,bbox_inches='tight')
plt.show(); print('Saved: results/latency_histograms.png')

# ── Figure 4 [NEW] — dispatch-gap chart ──────────────────────────────
_kern={'A: FP32':_nkern_a,'B: BF16':_nkern_b,'C: Split':_nkern_c,
       'D: Unified':_nkern_d,'E: Manual':_nkern_e}
_kern={k:v for k,v in _kern.items() if v is not None}
fig4,ax=plt.subplots(figsize=(8,4.5))
bars=ax.bar(list(_kern.keys()),list(_kern.values()),
            color=['#D85A30','#1D9E75','#3A7FC1','#8E44AD','#E8B12A'][:len(_kern)])
for b,v in zip(bars,_kern.values()):
    ax.text(b.get_x()+b.get_width()/2,b.get_height()+max(_kern.values())*0.02,
            str(v),ha='center',fontsize=9,fontweight='bold')
ax.set_ylabel('cudaLaunchKernel per spectrum')
ax.set_title('CPU Dispatch Gap — kernel launches per spectrum (bs=1)',fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/kernel_launch_comparison.png',dpi=150,bbox_inches='tight')
plt.show(); print('Saved: results/kernel_launch_comparison.png')

Saved: results/stage_comparison.png
Saved: results/throughput_comparison.png
Saved: results/latency_histograms.png
Saved: results/kernel_launch_comparison.png


In [13]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — Summary Report
# ═══════════════════════════════════════════════════════════════════════
_now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')

_best_key = min(ALL_TIMINGS, key=lambda k: ALL_TIMINGS[k][1]['total'])
_best_ms  = ALL_TIMINGS[_best_key][1]['total']

summary = f"""CASANOVO NAR PROFILING — CUDA DISPATCH GAP EXPERIMENT
Generated  : {_now}
Hardware   : {GPU_NAME} | {TOTAL_VRAM:.1f} GB VRAM | PyTorch {torch.__version__}
depthcharge: {_dc.__file__}  (branch: nar-flash-cuda-graph-compat)

OBJECTIVE (optimization O2)
  Close the residual CPU-dispatch gap at bs=1. Prior profiling showed only
  ~3.2 ms of the 22.2 ms baseline was real GPU compute; the best result so
  far (12.9 ms, compiled split + BF16 + Flash) still issued 138 kernel
  launches per spectrum. Three steps tested here:
    3B  torch._dynamo.explain  → identify remaining graph breaks
    4B  unified fullgraph=True → collapse enc+dec into ONE compiled region
    4C  manual CUDAGraph       → bypass torch.compile entirely

STEP 3B — GRAPH BREAK DIAGNOSIS
  encoder graph breaks : {_bk_enc}
  decoder graph breaks : {_bk_dec}
  unified graph breaks : {_bk_uni}   (graphs: {_gr_uni})
  Full break reasons + file:line → results/dynamo_explain.txt

STEP 4B — UNIFIED FULLGRAPH COMPILE
  fullgraph=True accepted : {'YES ✓' if FULLGRAPH_OK else 'NO'}
  {('' if FULLGRAPH_OK else 'error: ' + FULLGRAPH_ERR[:300])}

STEP 4C — MANUAL CUDA GRAPH CAPTURE
"""
for bs in BATCH_SIZES:
    summary += f"  bs={bs:<5} {'captured ✓' if MANUAL_OK.get(bs) else 'FAILED — '+str(MANUAL_ERR.get(bs,'not attempted'))[:80]}\n"

summary += f"""
DISPATCH METRICS (bs=1, {PROF_ACTIVE} profiled spectra, per spectrum)
{"Variant":<26}{"cudaLaunchKernel":>18}{"cudaGraphLaunch":>18}{"Regions":>10}
{"-"*72}
{"A: FP32 eager":<26}{str(_nkern_a):>18}{str(_ngraph_a):>18}{str(_nreg_a):>10}
{"B: BF16+Flash eager":<26}{str(_nkern_b):>18}{str(_ngraph_b):>18}{str(_nreg_b):>10}
{"C: Compiled SPLIT":<26}{str(_nkern_c):>18}{str(_ngraph_c):>18}{str(_nreg_c):>10}
{"D: Compiled UNIFIED":<26}{str(_nkern_d):>18}{str(_ngraph_d):>18}{str(_nreg_d):>10}
{"E: Manual CUDA Graph":<26}{str(_nkern_e):>18}{str(_ngraph_e):>18}{str(_nreg_e):>10}

ATTENTION KERNEL
  {_attn_a}
  {_attn_b}
  {_attn_c}
  {_attn_d}
  {_attn_e}

TIMING (real spectra, {N_TIMING_SPECTRA} per batch size, ms/spectrum)
{"Batch":>6}""" + ''.join(f'{LABELS[k][:14]:>16}' for k in ALL_TIMINGS) + '\n' + '-'*(6+16*len(ALL_TIMINGS)) + '\n'

for bs in BATCH_SIZES:
    summary += f'{bs:>6}' + ''.join(f'{ALL_TIMINGS[k][bs]["total"]:>16.2f}' for k in ALL_TIMINGS) + '\n'

summary += f'\nSPEEDUP vs FP32 baseline\n{"Batch":>6}' + ''.join(f'{LABELS[k][:14]:>16}' for k in ALL_TIMINGS) + '\n'
summary += '-'*(6+16*len(ALL_TIMINGS)) + '\n'
for bs in BATCH_SIZES:
    summary += f'{bs:>6}' + ''.join(
        f'{timing_fp32[bs]["total"]/max(ALL_TIMINGS[k][bs]["total"],0.001):>15.2f}x'
        for k in ALL_TIMINGS) + '\n'

summary += f"""
TAIL LATENCY (bs=1)
{"Variant":<28}{"mean":>10}{"p50":>10}{"p95":>10}
{"-"*58}
"""
for k in ALL_TIMINGS:
    t = ALL_TIMINGS[k][1]
    summary += f'{LABELS[k][:27]:<28}{t["total"]:>10.2f}{t["p50"]:>10.2f}{t["p95"]:>10.2f}\n'

summary += f"""
BEST bs=1 RESULT : {LABELS[_best_key]} — {_best_ms:.2f} ms  ({1000/_best_ms:.1f} spec/s)
  vs previous best (compiled split): {timing_compiled[1]['total']:.2f} ms
  improvement: {timing_compiled[1]['total']/max(_best_ms,0.001):.2f}×
  10 ms (100 Hz) target: {_tgt(_best_ms)}

GPU UTILIZATION / PEAK VRAM
  FP32 eager        : {gpu_util_fp32:.0f}% | {gpu_vram_fp32:.2f} GB
  BF16+Flash        : {gpu_util_bf16:.0f}% | {gpu_vram_bf16:.2f} GB
  Compiled split    : {gpu_util_comp:.0f}% | {gpu_vram_comp:.2f} GB
  Compiled unified  : {gpu_util_uni:.0f}% | {gpu_vram_uni:.2f} GB
  Manual CUDA Graph : {gpu_util_man:.0f}% | {gpu_vram_man:.2f} GB

INTERPRETATION GUIDE (for the mentor write-up)
  • If D (unified) ≈ C (split): the 2 regions were NOT the bottleneck —
    remaining latency is inductor/guard overhead, not fragmentation.
  • If D << C: fragmentation WAS the cost; the fix is to compile the
    full forward as one region rather than per-module. That is a
    casanovo-side inference-path change, not a depthcharge change.
  • If E (manual) << D: torch.compile's own bookkeeping (guards, static
    buffer copies) is the residual cost. A hand-captured graph is then
    the right production path for fixed-shape streaming inference.
  • If E fails to capture: something in the model still host-syncs or
    allocates dynamically — the exception names it, and that becomes the
    next source-level fix (same pattern as the spectra.py new_zeros fix).

ARTIFACTS
  results/dynamo_explain.txt        results/fullgraph_status.txt
  results/stage_comparison.png      results/throughput_comparison.png
  results/latency_histograms.png    results/kernel_launch_comparison.png
  results/trace_fp32.json  trace_bf16.json  trace_compiled.json
  results/trace_unified.json  trace_manual.json
  results/profiler_*.txt
  results/*_stage_bs1.csv  results/*_throughput.csv
  results/nar_summary.txt
"""
print(summary)
with open('results/nar_summary.txt','w') as f: f.write(summary)

print('\n── results/ ──')
for _f in sorted(os.listdir('results')):
    _fp = os.path.join('results', _f)
    print(f'  {_f:<48} {os.path.getsize(_fp)/1024:.1f} KB')
print('\nProfiling complete.')

CASANOVO NAR PROFILING — CUDA DISPATCH GAP EXPERIMENT
Generated  : 2026-08-18 14:29
Hardware   : NVIDIA L4 | 23.7 GB VRAM | PyTorch 2.7.1+cu128
depthcharge: /teamspace/studios/this_studio/depthcharge_changes/depthcharge/__init__.py  (branch: nar-flash-cuda-graph-compat)

OBJECTIVE (optimization O2)
  Close the residual CPU-dispatch gap at bs=1. Prior profiling showed only
  ~3.2 ms of the 22.2 ms baseline was real GPU compute; the best result so
  far (12.9 ms, compiled split + BF16 + Flash) still issued 138 kernel
  launches per spectrum. Three steps tested here:
    3B  torch._dynamo.explain  → identify remaining graph breaks
    4B  unified fullgraph=True → collapse enc+dec into ONE compiled region
    4C  manual CUDAGraph       → bypass torch.compile entirely

STEP 3B — GRAPH BREAK DIAGNOSIS
  encoder graph breaks : 0
  decoder graph breaks : 0
  unified graph breaks : 0   (graphs: 1)
  Full break reasons + file:line → results/dynamo_explain.txt

STEP 4B — UNIFIED FULLGRAPH COMPILE